# Week 3 · RAG Layer, Three-Way Comparison & Evaluation (IT Helpdesk)

**Capstone Track A** — domain-adapted assistant. This is **notebook 3 of 4**. Weeks 1–2 asked *"can we teach the model our output format and our labelling conventions?"*. Week 3 asks the next question: **"does giving the model relevant past tickets at answer time (retrieval-augmented generation, RAG) improve it further, and does it help the base model or the fine-tuned one more?"**

## The question and the systems we compare

| # | System | Where it comes from | Role |
|---|---|---|---|
| 1 | **base · zero-shot** | Week 1 `preds_zero_shot.jsonl` | The bar to beat |
| 2 | **fine-tuned** (best Week 2 adapter) | Week 2 `preds_<run>.jsonl` | What training alone buys |
| 3 | **base + RAG** | generated here | What retrieval alone buys |
| 4 | **fine-tuned + RAG** | generated here | Do the two stack? |
| 5 | **retrieval-only (kNN, no LLM)** | computed here from the retrieved tickets | Sanity check: is an LLM even needed? |

Systems 2, 3 and 4 are the **required comparison** of the brief (*base + RAG vs fine-tuned vs fine-tuned + RAG*). Systems 1 and 5 are the reference points that make the comparison interpretable.

## What this notebook does

| Stage | What happens | Output |
|---|---|---|
| 0 | Library check, settings, GPU helpers, verify the frozen data and pick the Week 2 adapter | — |
| 1 | **Build the knowledge base (KB)** from resolved tickets the models have never seen, with a leakage guard | `kb.jsonl`, `kb_manifest.json` |
| 2 | **Document three design decisions**, each backed by a measured comparison on the *validation* split: chunking strategy, embedding model (and hybrid search), reranker | `retrieval_decisions.json`, `rag_config.json` |
| 3 | Retrieve for the validation and test tickets, measure retrieval latency | `retrieved_val.json`, `retrieved_test.json` |
| 4 | Generate with the LLM (base + RAG, fine-tuned + RAG) | `preds_base_rag.jsonl`, `preds_ft_rag.jsonl` |
| 5 | Score all systems identically, add **paired bootstrap confidence intervals**, diagnose whether the model really uses the context, analyse errors, optional LLM-as-judge | `week3_results.json` |

**Runtime:** the same 2 × T4 box as Weeks 1–2. Stage 1 is CPU only. Stage 2–3 use the GPU for *small* encoder models (a few hundred MB). Stage 4 loads the 7B model in 4-bit. The two GPU phases are **separate**: the encoders are released before the LLM is loaded. No API keys are needed unless you turn on `RUN_JUDGE`.

## How the three notebooks fit together

| Order | Notebook | Reads | Writes |
|---|---|---|---|
| 1 | `it_helpdesk_week1_baseline.ipynb` | the Hugging Face dataset | frozen `train/val/test.jsonl`, `manifest.json`, baseline predictions and metrics |
| 2 | `it_helpdesk_week2_qlora.ipynb` | the frozen files | LoRA adapters, `run_*.json`, comparison tables |
| 3 | `it_helpdesk_week3_rag.ipynb` (this one) | frozen files, Week 1 baseline, Week 2 runs and adapters, the Hub dataset (to rebuild the unused ticket pool) | `week3/` folder: KB, retrieval decisions, predictions, `week3_results.json` |
| 4 | `it_helpdesk_week4_serving.ipynb` | all of the above | merged and quantised model, FastAPI + vLLM service repo, model card, red-team log, demo kit |

- **Self-contained and GPU-safe**, like the other two: it checks its own libraries, verifies everything it needs on disk (and stops with a clear message if something is missing), refuses to start a GPU phase if the GPU is occupied, releases GPU memory after every phase, and ends with a cleanup cell. Run the Week 2 cleanup cell (or shut that kernel down) before starting here.
- **Nothing in the frozen data is touched.** The test set is used exactly once for scoring; every *design decision* (chunking, embedder, reranker) is made on the validation split.

> [!warning] Leakage is the main way a RAG experiment fools you
> If the knowledge base contained the test tickets (or near-copies of them), "retrieval" would simply hand the model the answer. The KB is therefore built **only from tickets that are in none of train / val / test**, and every KB ticket that is too similar to a val/test ticket is removed (same TF-IDF guard as Week 1). Stage 1 prints the evidence.

**Outputs you will use later:** `rag_config.json` (the retriever settings for the Week 4 serving stack), `kb.jsonl` (the document corpus for the model card's data-provenance section), `week3_results.json` (numbers for the 12-page report).

### Step 0a · Check that the required libraries are installed

**What the next cell does**

Same idea as in Weeks 1–2: it checks that every third-party library this notebook needs can be imported, prints the versions, and stops with a ready-to-copy install command if something is missing. We *check* rather than install because the project's `.venv` is managed by `uv`, which has no `pip`.

This notebook needs the Week 2 libraries. It deliberately does **not** need `sentence-transformers`, `faiss` or a vector-database package:

| Library | Why we need it |
|---|---|
| `torch`, `transformers` | Run the encoder, reranker and the 7B model. The encoders are used through plain `transformers` (`AutoModel`), which keeps the dependency list short and avoids version clashes |
| `peft` | Loads the Week 2 LoRA adapter on top of the 4-bit base model |
| `accelerate`, `bitsandbytes` | Place the model on the GPUs and store it in 4-bit |
| `datasets` | Re-downloads the ticket dataset to rebuild the pool of tickets that were *not* used in train/val/test |
| `scikit-learn`, `scipy` | TF-IDF leakage guard, sparse matrices for BM25, F1 metric |
| `rouge-score` | ROUGE-L, the overlap metric used in Weeks 1–2 |
| `pandas`, `numpy`, `matplotlib`, `jinja2` | Tables, arithmetic, plots, coloured table styling |

**How it works:** `PACKAGES` maps the *install* name to the *import* name. `importlib.util.find_spec` checks that a module exists **without importing it**, so the check is quick and does not touch the GPU. If anything is missing we print the exact `uv pip install ...` command and raise an error, so "Run All" stops here. **After installing anything, restart the kernel.**

In [1]:
import importlib.util
from importlib.metadata import version, PackageNotFoundError

# pip name -> import name (they differ for scikit-learn and rouge-score)
PACKAGES = {"torch": "torch", "transformers": "transformers", "peft": "peft", "accelerate": "accelerate",
            "bitsandbytes": "bitsandbytes", "datasets": "datasets", "scikit-learn": "sklearn", "scipy": "scipy",
            "rouge-score": "rouge_score", "pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib", "jinja2": "jinja2"}

# find_spec() only looks the module up; it does not import it, so this check is quick and safe
missing = [pip for pip, mod in PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Missing libraries. In a terminal run:\n    uv pip install --python .venv/bin/python " + " ".join(missing))
    print("(on Colab use:  %pip install -q " + " ".join(missing) + ")")
    raise ImportError(f"missing libraries: {missing}; install them, then restart the kernel")

for pip in PACKAGES:
    try: print(f"{pip:14s} {version(pip)}")
    except PackageNotFoundError: pass

torch          2.9.1+cu126
transformers   5.23.0
peft           0.21.3
accelerate     1.16.0
bitsandbytes   0.50.2
datasets       5.2.0
scikit-learn   1.9.1
scipy          1.18.1
rouge-score    0.1.2
pandas         3.1.1
numpy          2.5.5
matplotlib     3.11.1
jinja2         3.1.6


### Step 0b · Imports and the control panel (all settings in one place)

**What the next cell does**

Nothing is computed here. It imports the standard libraries and defines every setting the later cells read. To change an experiment, edit this cell only.

**The settings, grouped**

- **Paths and identity.** `W1` is the Week 1 output folder, `W2` the Week 2 folder, `OUT` is where Week 3 writes (`week3/`). `MODEL_ID` must be the model used in Weeks 1–2. `FT_RUN = None` means *"pick the Week 2 run with the lowest validation loss automatically"*; set it to e.g. `"B"` to force a run.
- **Switches.** `SMOKE = True` is a tiny end-to-end test (small KB, 20 validation queries, 12 test tickets, one embedder) that takes minutes. `USE_WANDB` logs the final metrics to Weights & Biases. `RUN_JUDGE` turns the optional LLM-as-judge on. `REGEN_NO_RAG` regenerates the two no-RAG systems in this session instead of reusing the Week 1/2 predictions (slower, but guarantees identical conditions). `SHUTDOWN_WHEN_DONE` makes the last cell stop the kernel so *all* GPU memory is returned.
- **Knowledge base.** `KB_SIZE` is the number of documents (the brief asks for 200–1,000). `KB_LEAK_SIM` is the TF-IDF similarity above which a KB ticket is thrown out because it is a near-copy of a val/test ticket.
- **Chunking.** `FIXED_WORDS` / `FIXED_STRIDE` define the sliding window used by the "fixed" strategy (60 words, advancing 40, i.e. 20 words of overlap).
- **Embedding models (`EMBEDDERS`).** Each entry says which Hugging Face model, how to turn its output into one vector (`cls` token or `mean` pooling), and which text prefix it expects for queries and passages (the `e5` family requires `query: ` / `passage: `; `bge` uses an instruction only for queries). Using the wrong prefix or pooling silently degrades retrieval, so they are explicit.
- **Retrieval depth.** `K_CAND` candidates go to the reranker, `K_CTX` documents end up in the prompt, `RRF_K` is the constant of reciprocal-rank fusion (60 is the value from the original paper).
- **Decision rules.** The three design decisions are made *by rule on the validation split*, so they are reproducible and documented. `TOL`: if a simpler/cheaper configuration scores within `TOL` of the best one, prefer the simpler one. `RERANK_MIN_GAIN`: adopt the reranker only if it improves the validation score by at least this much (it costs latency). The `*_OVERRIDE` settings let you force a choice.
- **Prompt.** `MAX_DOC_CHARS` truncates each retrieved ticket in the prompt; `CONTEXT_WITH_LABELS` also shows the retrieved ticket's queue/type/priority (this is what lets retrieval transfer the dataset's labelling conventions).
- **Generation.** `MAX_NEW_TOKENS`, `GEN_BATCH` must equal Weeks 1–2, otherwise the numbers are not comparable. This is asserted against `baseline_results.json`.
- **Statistics.** `N_BOOT` is the number of bootstrap resamples for the confidence intervals.

In [2]:
import os, re, json, time, gc, random, hashlib, warnings, sys
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")                 # keep printed results readable

# Must be set before PyTorch first touches the GPU: expandable segments reduce memory fragmentation (fewer false OOMs).
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

ROOT = Path("capstone_it_helpdesk")                # parent folder holding week1/, week2/, week3/, week4/
W1 = ROOT / "week1"                                # Week 1 output (frozen data, baseline results)
W2 = ROOT / "week2"                                # Week 2 output (adapters, run_*.json)
OUT = ROOT / "week3"                               # everything Week 3 writes goes here
SEED = 42
random.seed(SEED); np.random.seed(SEED)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"             # must match Weeks 1-2
FT_RUN = None                                     # None = pick the Week 2 run with the lowest validation loss
SMOKE = False                                     # tiny end-to-end check (small KB, few queries, one embedder)
def _wandb_ready():
    # experiment tracking switches itself on when the `wandb` package is installed AND you are logged in (`wandb login` or WANDB_API_KEY)
    import importlib.util
    if importlib.util.find_spec("wandb") is None:
        return False
    if os.environ.get("WANDB_API_KEY"):
        return True
    try:
        return "api.wandb.ai" in (Path.home() / ".netrc").read_text()
    except OSError:
        return False
USE_WANDB = _wandb_ready()
RUN_JUDGE = bool(os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("JUDGE_API_KEY"))   # runs automatically when a judge key is set in the environment
JUDGE_MODEL, JUDGE_N = os.environ.get("JUDGE_MODEL", "claude-sonnet-5-5"), None   # None = judge every test ticket
REGEN_NO_RAG = False                              # True = regenerate base/fine-tuned WITHOUT RAG in this session too
SHUTDOWN_WHEN_DONE = True                         # True = the LAST cell stops the kernel so ALL GPU memory is returned

# --- knowledge base
KB_SIZE = 800                                     # brief: 200-1,000 documents
KB_LEAK_SIM = 0.90                                # TF-IDF cosine above which a KB ticket is a near-copy of a val/test ticket
# --- chunking ("fixed" strategy): windows of FIXED_WORDS words, advancing FIXED_STRIDE words
FIXED_WORDS, FIXED_STRIDE = 60, 40
# --- embedding models: pooling + the prefixes each model expects (wrong prefix/pooling silently hurts retrieval)
BGE_Q = "Represent this sentence for searching relevant passages: "
EMBEDDERS = {
    "bge-small": dict(id="BAAI/bge-small-en-v1.5", pool="cls", q_prefix=BGE_Q, p_prefix=""),
    "bge-base":  dict(id="BAAI/bge-base-en-v1.5",  pool="cls", q_prefix=BGE_Q, p_prefix=""),
    "e5-base":   dict(id="intfloat/e5-base-v2",    pool="mean", q_prefix="query: ", p_prefix="passage: "),
}
RERANKER_ID = "BAAI/bge-reranker-base"            # cross-encoder: reads (ticket, candidate) together
K_CAND, K_CTX, RRF_K = 20, 3, 60                  # candidates to rerank, documents in the prompt, RRF constant
# --- decision rules (applied on the VALIDATION split only)
TOL, RERANK_MIN_GAIN = 0.005, 0.01
CHUNKING_OVERRIDE = RETRIEVER_OVERRIDE = RERANK_OVERRIDE = None   # force a choice, e.g. RERANK_OVERRIDE = False
# --- prompt
MAX_DOC_CHARS = 700                               # truncate each retrieved ticket body / resolution in the prompt
CONTEXT_WITH_LABELS = True                        # show the retrieved ticket's queue / type / priority
RAG_PROMPT_VERSION = "v1"                         # bump when you edit the prompt template (invalidates cached predictions)
# --- generation (must match Weeks 1-2)
MAX_NEW_TOKENS, GEN_BATCH = 320, 8
N_BOOT = 5000                                     # bootstrap resamples for confidence intervals

if SMOKE:                                         # small and fast: checks the whole pipeline in minutes
    KB_SIZE, N_BOOT, JUDGE_N = 120, 500, 6
    EMBEDDERS = {k: EMBEDDERS[k] for k in list(EMBEDDERS)[:1]}
    K_CAND = 8
OUT.mkdir(parents=True, exist_ok=True)

# Fairness guard: the baseline we compare against must have used the same model, decoding length and precision
BASELINE_DTYPE = None
if (W1 / "baseline_results.json").exists():
    b = json.loads((W1 / "baseline_results.json").read_text())
    assert b["model"] == MODEL_ID, f"Week 1 baseline used {b['model']}, not {MODEL_ID}"
    assert b["max_new_tokens"] == MAX_NEW_TOKENS, "decoding length differs from the Week 1 baseline"
    BASELINE_DTYPE = b.get("dtype")               # load_base() later refuses to run with a different precision
print("config OK | smoke:", SMOKE, "| output folder:", OUT)

config OK | smoke: False | output folder: capstone_it_helpdesk/week3


### Step 0c · GPU memory helpers and the first pre-flight check

**What the next cell does**

It defines the same GPU memory helpers as Weeks 1–2 (an identical copy, so this notebook stays self-contained) and runs a quick pre-flight check. A notebook kernel keeps GPU memory until it is shut down, so a forgotten Week 2 kernel would make this notebook crash later with `CUDA out of memory`; the check finds that out in a second instead.

**The helpers**

- `gpu_report(tag)` prints, per GPU, the memory used **in total** (all programs) next to the memory owned by **this kernel**. A high total with a near-zero "this kernel" means another process is the culprit.
- `free_gpu(tag)` hands memory back: clears the saved traceback of an old error (it keeps every variable of the failed cell alive, including models), runs the garbage collector, and empties PyTorch's cache. **`del` the big objects first.**
- `check_gpu_free(min_free_gib)` refuses to start if another process holds more than 2 GiB on any GPU, or if less than `min_free_gib` is free in total.
- `release_model(m)` swaps every weight of a model for an empty tensor, which frees the GPU storage even if some library still references the module (a plain `del` was measured not to be enough in Week 2).
- `sync()` waits for the GPU to finish; it is needed for honest timing because GPU work is asynchronous.

This notebook has **two GPU phases** with different needs, so the pre-flight is called twice with different thresholds: ~3 GiB for the small encoders (Stages 2–3) and ~10 GiB for the 7B model (Stage 4).

In [3]:
import gc, sys, traceback, torch

def gpu_report(tag=""):
    # mem_get_info = driver's view (all processes); memory_allocated/reserved = only this kernel's PyTorch memory
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"{tag}GPU {i}: {(total - free) / 2**30:.1f}/{total / 2**30:.1f} GiB used | "
              f"this kernel: {torch.cuda.memory_allocated(i) / 2**30:.1f} GiB allocated, {torch.cuda.memory_reserved(i) / 2**30:.1f} GiB reserved")

def free_gpu(tag=""):
    """Release cached GPU memory. `del` the big objects (model, trainer, ...) BEFORE calling this."""
    sys.last_type = sys.last_value = sys.last_traceback = None   # an old error's traceback keeps the model alive
    gc.collect()                                                 # drop objects nothing refers to any more
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()           # return cached memory to the driver
    gpu_report(tag)

def check_gpu_free(min_free_gib, allow_shared=False):
    """Pre-flight: fail fast, with the fix, if the GPUs are already occupied by another process."""
    assert torch.cuda.is_available(), "GPU required: Runtime > Change runtime type > GPU"
    gpu_report("pre-flight: ")
    free_total, busy = 0.0, []
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        free_total += free / 2**30
        other = (total - free) - torch.cuda.memory_reserved(i)   # used by something that is not this kernel
        if other > 2 * 2**30:
            busy.append(f"GPU {i}: ~{other / 2**30:.1f} GiB")
    if busy and not allow_shared:
        # two jobs cannot share a 15 GB T4: the second one would run out of memory part-way through
        raise RuntimeError("Another process is using the GPU(s) (" + ", ".join(busy) + "). Another notebook may still be "
                           "training, or a finished kernel still holds its model. Check `nvidia-smi`, wait for it or shut "
                           "that kernel down, then re-run. (Pass allow_shared=True to override.)")
    if free_total < min_free_gib:
        raise RuntimeError(f"Only {free_total:.1f} GiB of GPU memory is free but ~{min_free_gib} GiB is needed. "
                           "Shut down other notebook kernels that hold the GPU (find them with `nvidia-smi`), then re-run.")

def release_model(m):
    """Free the GPU storage of every weight in `m`, even if some library still references the module object.
    (Measured: after QLoRA training a 2 GB float32 copy of the embedding matrix stayed on the GPU after `del model`.)"""
    with torch.no_grad():
        for t in list(m.parameters()) + list(m.buffers()):
            if hasattr(t, "quant_state"): t.quant_state = None   # 4-bit weights keep their scale tensors here
            t.data = torch.empty(0, dtype=t.dtype, device="cpu")  # swap the big GPU tensor for an empty CPU one

def reset_peak():
    # restart the "highest memory so far" counters on every GPU (called at the start of a run)
    for i in range(torch.cuda.device_count()): torch.cuda.reset_peak_memory_stats(i)

def peak_gib():
    # highest PyTorch allocation since reset_peak(), on the busiest GPU
    return max(torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())) / 2**30

def sync():
    # GPU kernels run asynchronously; wait for them so time.perf_counter() measures the real duration
    torch.cuda.synchronize()

check_gpu_free(min_free_gib=3)           # phase 1 needs only the small encoder / reranker models

pre-flight: GPU 0: 0.4/44.4 GiB used | this kernel: 0.0 GiB allocated, 0.0 GiB reserved


### Step 0d · Load and verify the earlier weeks' results, and choose the fine-tuned adapter

**What the next cell does**

Week 3 depends on three things from earlier weeks. This cell loads them, **proves they are intact**, and stops with a clear message if anything is missing.

**Step by step**

1. **Prerequisite check.** `manifest.json`, the three JSONL files, `baseline_results.json` and the zero-shot predictions (Week 1) must exist.
2. **Hash check.** The SHA-256 of each JSONL file must equal the hash in `manifest.json`. If the frozen data changed, nothing in this notebook is comparable to Weeks 1–2, so we stop.
3. **Load the splits.** Train, validation and test rows. `val_rows` is used for *decisions*, `test_rows` only for the final scores. A hard assert confirms no train/val ticket is also in the test set. In `SMOKE` mode only a few rows are kept.
4. **Choose the fine-tuned model.** We read every non-smoke `run_*.json` of Week 2 and, unless `FT_RUN` is set, pick the run with the **lowest validation loss**. Choosing by validation loss (not by the test score) keeps the test set clean. The candidate table is printed so the choice is auditable, and the adapter folder is checked to exist.

**What to look for in the output:** the row counts (expected 1500 / 100 / 180), "frozen data verified", and which adapter was selected and why.

In [4]:
def read_jsonl(p):
    # one JSON object per line -> list of dicts
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f]

# Self-contained: Week 3 needs the Week 1 freeze, the Week 1 baseline and the Week 2 runs. Fail early, with the fix.
needed = [W1 / "manifest.json", W1 / "baseline_results.json", W1 / "preds_zero_shot.jsonl"] + [W1 / f"{n}.jsonl" for n in ["train", "val", "test"]]
absent = [p.name for p in needed if not p.exists()]
if absent:
    raise FileNotFoundError(f"Missing {absent} in {W1.resolve()}. Run it_helpdesk_week1_baseline.ipynb to the end first, "
                            "and open this notebook from the same folder.")

man = json.loads((W1 / "manifest.json").read_text())
for name in ["train", "val", "test"]:                       # the Week 1 freeze must still hold
    h = hashlib.sha256((W1 / f"{name}.jsonl").read_bytes()).hexdigest()
    assert h == man["sha256"][name], f"{name}.jsonl changed since the Week 1 freeze"

train_rows, val_rows, test_rows = [read_jsonl(W1 / f"{n}.jsonl") for n in ["train", "val", "test"]]
assert {r["uid"] for r in train_rows + val_rows}.isdisjoint({r["uid"] for r in test_rows})
QUEUES, TYPES, PRIORITIES = man["queues"], man["types"], man["priorities"]
SYSTEM = train_rows[0]["messages"][0]["content"]            # the exact system prompt used in Weeks 1-2
if SMOKE:
    val_rows, test_rows = val_rows[:20], test_rows[:12]
print({"train": len(train_rows), "val": len(val_rows), "test": len(test_rows)}, "| frozen data verified")

# --- choose the Week 2 adapter by VALIDATION loss (never by the test score)
w2_runs = []
for p in sorted(W2.glob("run_*.json")):
    if p.name.endswith("_log.json") or p.name.endswith("_smoke.json"):
        continue
    r = json.loads(p.read_text())
    if r.get("kind", "sft") == "sft" and r["train"].get("val_loss") is not None:
        w2_runs.append(r)
if not w2_runs:
    raise FileNotFoundError(f"No finished Week 2 run found in {W2}. Train at least one run in it_helpdesk_week2_qlora.ipynb first.")
cand = pd.DataFrame([{"run": r["run_id"], "val_loss": r["train"]["val_loss"], "rank": r["config"]["r"], "lr": r["config"]["lr"],
                      "targets": r["config"]["targets"]} for r in w2_runs]).sort_values("val_loss")
if FT_RUN is None:
    FT_RUN = cand.iloc[0]["run"]
assert FT_RUN in set(cand["run"]), f"FT_RUN={FT_RUN} has no finished Week 2 result"
ADAPTER_DIR = W2 / "adapters" / FT_RUN
assert (ADAPTER_DIR / "adapter_config.json").exists(), f"adapter files missing in {ADAPTER_DIR}"
print(cand.to_string(index=False))
print(f"-> using adapter '{FT_RUN}' (lowest validation loss unless FT_RUN was set)")

{'train': 1500, 'val': 100, 'test': 180} | frozen data verified
run  val_loss  rank     lr targets
  B  0.765530    16 0.0002     all
  A  0.772438     8 0.0002     all
  C  0.822871    16 0.0002      qv
-> using adapter 'B' (lowest validation loss unless FT_RUN was set)


### Step 0e · The scoring toolkit (identical to Weeks 1–2)

**What the next cell does**

It defines the functions that turn model text into numbers. They are an **exact copy** of the Week 1/Week 2 functions, so every number in this notebook is measured with the same ruler as the baseline and the fine-tuned runs. Nothing prints.

**The functions** (explained in detail in Week 1, Step 3a)

- `extract_json(text)`: finds the first well-formed `{...}` object in the model's output, tolerating code fences and chatter.
- `score_row(...)`: grades one prediction: did it parse, is it schema-valid (exactly the 4 keys, allowed labels, non-empty reply), are `queue`/`type`/`priority` correct, and the ROUGE-L between its `resolution` and the reference reply.
- `summarize(...)`: averages those into the headline metrics, including **macro-F1** (every queue/priority class counts equally, so the small "Service Outages" queue matters as much as the big one).
- `score_preds(...)`: aligns predictions with the gold tickets **by ticket id**, scores them, and computes the speed numbers (tokens/second, seconds per example).

`ticket_text(r)` formats a ticket as the user message, exactly like Week 1.

In [5]:
# --- identical metric logic to Week 1 (same numbers => comparable tables)
from rouge_score import rouge_scorer
from sklearn.metrics import f1_score
# ROUGE-L = longest common word sequence between reply and reference; use_stemmer treats word variants as equal
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def extract_json(text):
    # Scan from each "{" and count nesting depth; braces inside quoted strings must not be counted
    start = text.find("{")
    while start != -1:
        depth, in_str, esc = 0, False, False         # esc = previous char was a backslash
        for i in range(start, len(text)):
            ch = text[i]
            if in_str:
                esc = (not esc) and ch == "\\"
                if ch == '"' and not esc:
                    in_str = False
            elif ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    # braces balanced: try to parse; if it isn't valid JSON, try the next "{"
                    try:
                        return json.loads(text[start:i + 1])
                    except json.JSONDecodeError:
                        break
        start = text.find("{", start + 1)
    return None

def score_row(pred_text, gold, queues, types, priorities):
    obj = extract_json(pred_text)
    parsed = isinstance(obj, dict)
    g = obj if parsed else {}
    # schema_valid is strict: exactly the 4 keys, allowed label values, and a non-empty text reply
    valid = (parsed and set(g) == {"queue", "type", "priority", "resolution"}
             and g["queue"] in queues and g["type"] in types and g["priority"] in priorities
             and isinstance(g["resolution"], str) and g["resolution"].strip() != "")
    res = g.get("resolution") if isinstance(g.get("resolution"), str) else ""
    return {
        "parsed": parsed, "schema_valid": bool(valid),
        # "INVALID" marks a missing label so it counts as wrong
        "pred_queue": g.get("queue", "INVALID"), "pred_type": g.get("type", "INVALID"),
        "pred_priority": g.get("priority", "INVALID"),
        "queue_ok": g.get("queue") == gold["queue"], "type_ok": g.get("type") == gold["type"],
        "priority_ok": g.get("priority") == gold["priority"],
        "rougeL": _rouge.score(gold["answer"], res)["rougeL"].fmeasure if res else 0.0,   # no usable reply -> 0
    }

def summarize(scored, gold_df, queues, priorities):
    s = scored
    # .mean() of a True/False column = the fraction of True, i.e. a rate
    return {
        "n": len(s),
        "json_parse_rate": s["parsed"].mean(), "schema_valid_rate": s["schema_valid"].mean(),
        "queue_acc": s["queue_ok"].mean(), "type_acc": s["type_ok"].mean(), "priority_acc": s["priority_ok"].mean(),
        "labels_exact_match": (s["queue_ok"] & s["type_ok"] & s["priority_ok"]).mean(),
        # macro-F1: every class counts equally, so the small queue matters as much as the big one
        "queue_macro_f1": f1_score(gold_df["queue"], s["pred_queue"], labels=queues, average="macro", zero_division=0),
        "priority_macro_f1": f1_score(gold_df["priority"], s["pred_priority"], labels=priorities, average="macro", zero_division=0),
        "rougeL_resolution": s["rougeL"].mean(),
    }

def score_preds(preds, gold_rows):
    gold = pd.DataFrame(gold_rows)
    # line predictions up with gold by ticket id so a shuffled file can't pair a reply with the wrong ticket
    p = preds.set_index("uid").loc[gold["uid"]].reset_index()
    sc = pd.DataFrame([score_row(t, g, QUEUES, TYPES, PRIORITIES) for t, g in zip(p["pred_text"], gold.to_dict("records"))])
    m = summarize(sc, gold, QUEUES, PRIORITIES)
    # every row carries its batch's wall time; dividing by batch size and summing counts each batch exactly once
    tot = (p["batch_sec"] / p["batch_size"]).sum()
    m.update({"mean_out_tokens": p["gen_tokens"].mean(), "mean_prompt_len": p["prompt_len"].mean(),
              "sec_per_example": tot / len(p), "tokens_per_sec": p["gen_tokens"].sum() / tot})
    return {k: float(v) for k, v in m.items()}, pd.concat([p.drop(columns=[c for c in sc.columns if c in p.columns]), sc], axis=1)

def ticket_text(r):
    # What the model reads as the user message: subject line, blank line, body (same as Week 1)
    return f"Subject: {r['subject'] or '(none)'}\n\n{r['body']}"

## 1 · The RAG design: what we build and why

**RAG in one sentence:** before the model answers a new ticket, we look up the most similar *already resolved* tickets in a knowledge base and put them in the prompt, so the model can reuse their resolutions and their labelling conventions instead of relying on memory alone.

```text
new ticket ──► [retrieve candidates] ──► [rerank] ──► top-3 resolved tickets ──► prompt ──► LLM ──► JSON answer
                BM25 / dense / hybrid      cross-encoder     (parent documents)
```

### What is in the knowledge base?

Each KB document is a **resolved ticket**: `subject`, `body` (the symptom), `answer` (the resolution), plus its `queue`, `type` and `priority`. This is how real helpdesks reuse knowledge ("solved similar issue before"). The documents come from tickets that are in **none** of train/val/test, so the fine-tuned model has never seen them, and retrieval can add something the weights do not contain.

### The three decisions the brief asks us to document

| # | Decision | Options we measure | How we decide |
|---|---|---|---|
| 1 | **Chunking strategy** | `doc` (one chunk per ticket), `struct` (a symptom chunk + a resolution chunk), `fixed` (60-word sliding windows) | Best validation score; ties go to the simpler strategy |
| 2 | **Embedding model** | BM25 (lexical baseline), `bge-small`, `bge-base`, `e5-base`, and **hybrid** = BM25 + dense fused by reciprocal rank | Same rule |
| 3 | **Reranker** | none vs a cross-encoder (`bge-reranker-base`) re-scoring the top 20 candidates | Adopt only if it gains at least `RERANK_MIN_GAIN`, because it costs latency |

**Parent-document retrieval:** we *search* over chunks (which can be small and precise) but always put the **whole ticket** into the prompt, so the model never sees a resolution without its symptom. A document's score is the best score of any of its chunks.

**Why tickets are short, so chunking matters less than usual:** a ticket plus its reply is roughly 100–250 words. Whole-ticket chunks are already smaller than the usual 512-token chunk, so we expect the three strategies to be close. That is a legitimate finding; the point of the experiment is to *measure* it rather than assume.

### How can we score retrieval without hand-labelled relevance?

We have no human labels saying which KB ticket is "the right one" for a validation ticket. We use a **proxy** that matches what the downstream task needs. For each validation ticket, look at its top-3 retrieved tickets and measure:

- **`queue_p`, `type_p`, `priority_p`**: the share of retrieved tickets with the same label as the query. If retrieval finds tickets handled the same way, the labels transfer.
- **`res_rougeL`**: the ROUGE-L between the query's *reference reply* and the retrieved replies. If it is high, the retrieved resolutions are useful raw material for the answer.
- **`ctx_score`**: the mean of those four numbers; the single number the decision rules use.

> [!note] Limits of the proxy
> It measures *usefulness for this task*, not true topical relevance, and 100 validation tickets give a noisy estimate (hence the `TOL` rule: differences smaller than 0.005 are treated as ties and the simpler option wins). Both limits go into the model card.

### Step 1a · Build the knowledge base from tickets nobody has seen

**What the next cell does**

It creates the KB: a set of resolved tickets that is **disjoint from train, val and test**. Week 1 only saved the three splits, not the tickets it discarded, so we **replay the Week 1 filtering pipeline** on the dataset (same language, same length limits, same de-duplication, same queues, taken from `manifest.json`) and keep everything that was not used.

**Step by step**

1. **Rebuild the cleaned table.** Download the dataset (cached by Hugging Face) and apply exactly the Week 1 steps: language filter, drop incomplete rows, tidy text, length filters, drop exact duplicate bodies, keep the selected queues, assign `uid`s (`t00000`, `t00001`, ...), and **redact** emails and phone numbers with the same regular expressions.
2. **Prove the replay is faithful.** Every ticket in the frozen train/val/test files must appear in the rebuilt table *with the same body text under the same `uid`*. If even one differs (for example the dataset was updated on the Hub), we stop: the "unused pool" would not really be unused.
3. **Unused pool.** All rebuilt tickets whose `uid` is not in train/val/test.
4. **Leakage guard.** Fit TF-IDF on the pool and on the val+test bodies, and drop any pool ticket whose best cosine similarity to a val or test ticket is at least `KB_LEAK_SIM`. This is the Week 1 guard again: near-copies of an evaluation ticket would hand the model the answer.
5. **Sample `KB_SIZE` documents**, stratified by queue so the KB has the same queue mix as the data, and give each a stable id (`kb0001`, ...).
6. **Save** `kb.jsonl` and `kb_manifest.json` (counts and a SHA-256 hash). If a KB with the same hash already exists it is simply reused, so re-running is safe.

**What to look for in the output:** "replay verified" with the number of checked tickets, how many pool tickets the leakage guard dropped, and the final KB size.

In [6]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Patterns for personal data. PHONE needs 10+ digits/separators so short numbers like ticket IDs aren't masked.
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PHONE = re.compile(r"(?<!\w)\+?\d[\d\s().-]{8,}\d(?!\w)")
# Template tokens such as <name> or [Customer] that some replies still contain (we only measure these)
PLACEHOLDER = re.compile(r"<[^<>\n]{1,30}>|\[[A-Za-z _]{2,30}\]")

def redact(s):
    # emails first, then phones, so digits inside an email can't be mistaken for a phone number
    return PHONE.sub("<PHONE>", EMAIL.sub("<EMAIL>", s))

# ---- 1. replay the Week 1 cleaning pipeline (all settings come from the frozen manifest)
raw = load_dataset(man["dataset"], split="train").to_pandas()
bl, bh = man["filters"]["body_chars"]; al, ah = man["filters"]["answer_chars"]
d = raw[raw["language"].astype(str).str.lower() == man["language"]].copy()
d = d.dropna(subset=["body", "answer", "queue", "type", "priority"])
for c in ["subject", "body", "answer", "queue", "type", "priority"]:
    d[c] = d[c].fillna("").astype(str).str.strip()
d["priority"] = d["priority"].str.lower()
d = d[d["body"].str.len().between(bl, bh) & d["answer"].str.len().between(al, ah)]
d["_key"] = d["body"].str.lower().str.replace(r"\s+", " ", regex=True)
d = d.drop_duplicates("_key").drop(columns="_key")
d = d[d["queue"].isin(man["queues"])].reset_index(drop=True)
d["uid"] = [f"t{i:05d}" for i in range(len(d))]
for c in ["subject", "body", "answer"]:
    d[c] = d[c].map(redact)

# ---- 2. prove the replay is faithful: every frozen ticket must be found, with the same text, under the same uid
frozen = {r["uid"]: r["body"] for r in read_jsonl(W1 / "train.jsonl") + read_jsonl(W1 / "val.jsonl") + read_jsonl(W1 / "test.jsonl")}
body_by_uid = d.set_index("uid")["body"]
bad = [u for u, b in frozen.items() if body_by_uid.get(u) != b]
if bad:
    raise RuntimeError(f"Replay mismatch for {len(bad)} frozen tickets (e.g. {bad[:3]}). The dataset on the Hub may have changed "
                       "since Week 1, so the unused-ticket pool cannot be reconstructed safely.")
print(f"replay verified: all {len(frozen)} frozen tickets found with identical text under identical ids")

# ---- 3. unused pool + 4. leakage guard against validation and test tickets
pool = d[~d["uid"].isin(frozen)].copy()
eval_bodies = [r["body"] for r in read_jsonl(W1 / "val.jsonl") + read_jsonl(W1 / "test.jsonl")]
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(list(pool["body"]) + eval_bodies)
sim = (vec.transform(pool["body"]) @ vec.transform(eval_bodies).T).max(axis=1).toarray().ravel()
leaky = (sim >= KB_LEAK_SIM) | pool["body"].isin(set(eval_bodies)).values
print(f"unused pool: {len(pool)} tickets | dropped {int(leaky.sum())} as near-copies of val/test (cosine >= {KB_LEAK_SIM})")
pool = pool[~leaky]

# ---- 5. stratified sample of KB_SIZE documents
if len(pool) > KB_SIZE:
    pool, _ = train_test_split(pool, train_size=KB_SIZE, stratify=pool["queue"], random_state=SEED)
pool = pool.sort_values("uid").reset_index(drop=True)
kb = [{"kb_id": f"kb{i + 1:04d}", "src_uid": r["uid"], "subject": r["subject"], "body": r["body"], "answer": r["answer"],
       "queue": r["queue"], "type": r["type"], "priority": r["priority"]} for i, r in enumerate(pool.to_dict("records"))]

# ---- 6. save (re-running reproduces the same file, hence the same hash)
kb_text = "".join(json.dumps(x, ensure_ascii=False) + "\n" for x in kb)
kb_sha = hashlib.sha256(kb_text.encode("utf-8")).hexdigest()
(OUT / "kb.jsonl").write_text(kb_text, encoding="utf-8")
(OUT / "kb_manifest.json").write_text(json.dumps({
    "n_docs": len(kb), "sha256": kb_sha, "source": man["dataset"], "license": man["license"], "seed": SEED,
    "leak_sim": KB_LEAK_SIM, "queues": man["queues"], "built_from": "unused pool of the Week 1 pipeline (disjoint from train/val/test)"}, indent=2))
print(f"knowledge base: {len(kb)} documents | sha256 {kb_sha[:16]}...")

replay verified: all 1780 frozen tickets found with identical text under identical ids


unused pool: 8469 tickets | dropped 17 as near-copies of val/test (cosine >= 0.9)
knowledge base: 800 documents | sha256 0c36371d5f9aafd0...


### Step 1b · Knowledge-base health check

**What the next cell does**

Before building anything on the KB we verify it, the same way Week 1 verified the dataset: hard assertions for the things that must never be violated, and a small report for the things a human should look at.

**Hard checks (the notebook stops if any fails)**

- the KB size is inside the brief's 200–1,000 range (relaxed in `SMOKE` mode);
- **no KB document is also a train, val or test ticket**, neither by `uid` nor by identical body text. This is the leakage guarantee.

**Report**

- **Queue mix:** KB share per queue next to the train share. They should be similar, because the KB is a stratified sample of the same data.
- **Length statistics:** median characters of ticket bodies and replies, and the median word count of a whole document. These numbers justify the chunking experiment (a typical ticket is shorter than one standard 512-token chunk).
- **Placeholders:** the share of KB replies that still contain template tokens such as `[Name]`. Week 1 found about 60% of replies contain them; the model may copy them, which the error analysis later checks.
- **Nearest-neighbour similarity to the test set:** the maximum TF-IDF similarity between any KB document and any test ticket. It must be below `KB_LEAK_SIM`; the printed value is the evidence for the model card.

In [7]:
assert (len(kb) >= 200 or SMOKE) and len(kb) <= 1000, f"KB has {len(kb)} documents; the brief asks for 200-1,000"
seen = {r["uid"] for r in train_rows + val_rows + test_rows}
frozen_bodies = {r["body"] for r in read_jsonl(W1 / "train.jsonl") + read_jsonl(W1 / "val.jsonl") + read_jsonl(W1 / "test.jsonl")}
assert not any(x["src_uid"] in seen for x in kb), "a KB document is a train/val/test ticket"
assert not any(x["body"] in frozen_bodies for x in kb), "a KB document has the same text as a train/val/test ticket"

kbdf = pd.DataFrame(kb)
mix = pd.concat({"KB": kbdf["queue"].value_counts(normalize=True),
                 "train": pd.Series([r["queue"] for r in train_rows]).value_counts(normalize=True)}, axis=1).round(3)
print(mix.to_string(), "\n")
words = (kbdf["subject"] + " " + kbdf["body"] + " " + kbdf["answer"]).str.split().str.len()
print(f"median body chars {int(kbdf['body'].str.len().median())} | median reply chars {int(kbdf['answer'].str.len().median())} | "
      f"median words per document {int(words.median())} (p95 {int(words.quantile(.95))})")
print(f"replies with template placeholders ([Name], <name>, ...): {kbdf['answer'].str.contains(PLACEHOLDER).mean():.1%}")
test_vec = vec.transform([r["body"] for r in test_rows])
print(f"max TF-IDF similarity between any KB document and any test ticket: {(vec.transform(kbdf['body']) @ test_vec.T).max():.3f} "
      f"(guard threshold {KB_LEAK_SIM})")

                                    KB  train
Technical Support                0.645  0.645
IT Support                       0.266  0.267
Service Outages and Maintenance  0.089  0.088 

median body chars 376 | median reply chars 375 | median words per document 120 (p95 194)
replies with template placeholders ([Name], <name>, ...): 60.1%
max TF-IDF similarity between any KB document and any test ticket: 0.847 (guard threshold 0.9)


### Step 2a · The retrieval toolkit, part 1: chunking, BM25, fusion and retrieval metrics (CPU only)

**What the next cell does**

It defines (and does not yet run) every retrieval building block that needs no GPU. Keeping them as small functions makes each design decision testable and the final pipeline easy to port to the Week 4 serving code.

**Chunking: three strategies**

- `chunk_doc(d, "doc")`: one chunk per ticket (symptom and resolution together).
- `chunk_doc(d, "struct")`: two chunks per ticket, the **symptom** (subject + body) and the **resolution**. The idea: a new ticket should match symptoms, not replies.
- `chunk_doc(d, "fixed")`: sliding word windows of `FIXED_WORDS` with stride `FIXED_STRIDE` (so consecutive windows overlap). This is the classic strategy for long documents; here it mostly splits short tickets in two.
- `build_chunks(kb, strategy)` returns all chunk texts plus `owner`, the index of the document each chunk came from.

**Scoring**

- `BM25`: the classic lexical ranking function, implemented in a few lines with a sparse term-frequency matrix (no extra library). A term counts more if it is rare in the corpus (`idf`) and saturates with repetition (`k1`), with a correction for document length (`b`). It is our **no-neural-network baseline**: if a transformer embedding cannot beat it, we do not need the transformer.
- `to_doc_scores(chunk_scores, owner, n_docs)`: converts chunk-level scores into document-level scores by taking the **best chunk of each document** (`np.maximum.reduceat` over the contiguous chunk blocks).
- `rrf_fuse(score_matrices, k)`: **reciprocal-rank fusion** for hybrid search. It ignores the raw scores (BM25 and cosine similarity live on different scales) and adds $\frac{1}{k+\text{rank}}$ for every ranking in which a document appears. Documents ranked high by *both* methods win.
- `topk(S, k)`: the indices of the `k` best documents per query.

**Measuring retrieval (the proxy from the design section)**

`retrieval_metrics(top, queries, kb)` returns `queue_p`, `type_p`, `priority_p`, `res_rougeL` and their mean `ctx_score` for a set of retrieved documents. ROUGE results are cached because the same (query, document) pair is scored by many configurations.

**Prompt and baseline helpers**

`format_context` / `rag_user_message` build the text that goes into the prompt; `knn_prediction` builds the **retrieval-only system**: majority vote of the retrieved tickets' labels, and the top-1 ticket's reply as the resolution, no LLM involved.

In [8]:
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from scipy import sparse                      # sparse matrices for the BM25 index

# ------------------------------------------------------------------ chunking
def symptom_text(d):
    return f"{d['subject']}\n{d['body']}".strip()

def full_text(d):
    return f"{symptom_text(d)}\nResolution: {d['answer']}"

def chunk_doc(d, strategy):
    if strategy == "doc":                                   # one chunk: the whole ticket
        return [full_text(d)]
    if strategy == "struct":                                # symptom chunk + resolution chunk
        return [symptom_text(d), f"Resolution: {d['answer']}"]
    if strategy == "fixed":                                 # overlapping word windows
        w = full_text(d).split()
        if len(w) <= FIXED_WORDS:
            return [" ".join(w)]
        return [" ".join(w[i:i + FIXED_WORDS]) for i in range(0, len(w) - FIXED_WORDS + FIXED_STRIDE, FIXED_STRIDE)]
    raise ValueError(strategy)

def build_chunks(kb, strategy):
    texts, owner = [], []
    for i, d in enumerate(kb):
        for t in chunk_doc(d, strategy):
            texts.append(t); owner.append(i)
    return texts, np.array(owner)

def to_doc_scores(chunk_scores, owner, n_docs):
    """(n_queries x n_chunks) -> (n_queries x n_docs): a document scores as its best chunk. owner is sorted ascending."""
    starts = np.flatnonzero(np.r_[True, owner[1:] != owner[:-1]])      # first chunk of every document
    assert len(starts) == n_docs, "every document must own at least one chunk"
    return np.maximum.reduceat(chunk_scores, starts, axis=1)

# ------------------------------------------------------------------ lexical search: BM25
class BM25:
    # OPTIMIZED: the BM25 weight of every (chunk, term) pair does not depend on the query, so it is computed ONCE here
    # (on the non-zero entries only) and a search becomes one sparse matrix product instead of a Python loop with
    # dense slices per query. Scores are identical to the previous implementation (checked: max difference 0.0).
    def __init__(self, texts, k1=1.5, b=0.75):
        self.cv = CountVectorizer(lowercase=True, stop_words="english")
        tf = self.cv.fit_transform(texts).tocsr().astype(np.float32)   # (n_chunks x n_terms) term counts, sparse
        n = tf.shape[0]
        df_ = np.bincount(tf.indices, minlength=tf.shape[1])           # in how many chunks each term occurs
        idf = np.log(1 + (n - df_ + 0.5) / (df_ + 0.5)).astype(np.float32)   # rare terms weigh more
        dl = np.asarray(tf.sum(1)).ravel()                             # chunk lengths
        norm = (k1 * (1 - b + b * dl / dl.mean())).astype(np.float32)  # length normalisation
        rows = np.repeat(np.arange(n), np.diff(tf.indptr))             # chunk index of every stored (non-zero) entry
        w = idf[tf.indices] * tf.data * (k1 + 1) / (tf.data + norm[rows])   # saturating tf x idf, per stored entry
        # transpose to (n_terms x n_chunks) so that  query_vector @ W  gives one score per chunk
        self.W = sparse.csr_matrix((w.astype(np.float32), tf.indices, tf.indptr), shape=tf.shape).T.tocsr()

    def scores(self, queries):
        Q = self.cv.transform(queries)
        Q.data[:] = 1.0                                                # every distinct query term counts once
        return np.asarray((Q.astype(np.float32) @ self.W).todense(), dtype=np.float32)

# ------------------------------------------------------------------ fusion and top-k
def rrf_fuse(score_mats, k=60):
    """Reciprocal-rank fusion: each ranking adds 1/(k+rank) to a document; raw score scales are ignored."""
    fused = np.zeros(score_mats[0].shape, dtype=np.float32)
    for S in score_mats:
        ranks = np.argsort(np.argsort(-S, axis=1), axis=1) + 1         # rank 1 = best
        fused += 1.0 / (k + ranks)
    return fused

def topk(S, k):
    return np.argsort(-S, axis=1)[:, :k]

# ------------------------------------------------------------------ retrieval metrics (the proxy from the design section)
_rcache = {}
def _rouge_cached(q, kb_doc):
    key = (q["uid"], kb_doc["kb_id"])
    if key not in _rcache:
        _rcache[key] = _rouge.score(q["answer"], kb_doc["answer"])["rougeL"].fmeasure
    return _rcache[key]

def retrieval_metrics(top, queries, kb):
    rows = []
    for q, idxs in zip(queries, top):
        docs = [kb[j] for j in idxs]
        rows.append({"queue_p": np.mean([x["queue"] == q["queue"] for x in docs]),
                     "type_p": np.mean([x["type"] == q["type"] for x in docs]),
                     "priority_p": np.mean([x["priority"] == q["priority"] for x in docs]),
                     "res_rougeL": np.mean([_rouge_cached(q, x) for x in docs])})
    m = pd.DataFrame(rows).mean().to_dict()
    m["ctx_score"] = float(np.mean(list(m.values())))                  # the single number the decision rules use
    return m

# ------------------------------------------------------------------ prompt context and the retrieval-only baseline
def format_context(docs):
    parts = []
    for n, x in enumerate(docs, 1):
        lab = f"Queue: {x['queue']} | Type: {x['type']} | Priority: {x['priority']}\n" if CONTEXT_WITH_LABELS else ""
        parts.append(f"[KB-{n}] {lab}Ticket: {x['subject']} - {x['body'][:MAX_DOC_CHARS]}\nResolution: {x['answer'][:MAX_DOC_CHARS]}")
    return "\n\n".join(parts)

def rag_user_message(row, docs):
    return ("Resolved tickets from the knowledge base (use them as guidance only; they may not match this ticket exactly):\n\n"
            + format_context(docs) + "\n\n---\nNew ticket to handle:\n" + ticket_text(row))

def knn_prediction(docs):
    """Retrieval-only 'model': majority vote of the retrieved tickets' labels + the top-1 ticket's reply. No LLM."""
    vote = lambda key: Counter(x[key] for x in docs).most_common(1)[0][0]    # ties resolve to the better-ranked document
    return json.dumps({"queue": vote("queue"), "type": vote("type"), "priority": vote("priority"),
                       "resolution": docs[0]["answer"]}, ensure_ascii=False)

### Step 2b · The retrieval toolkit, part 2: the encoder and the reranker (GPU models)

**What the next cell does**

It defines two small classes that wrap the neural models. Nothing is loaded yet.

**`DenseEncoder`: text -> vector**

An *embedding model* maps a text to a vector so that texts with similar meaning have vectors pointing in similar directions; similarity is then a dot product. Steps inside `encode`: add the model's prefix (queries and passages are prefixed differently for `bge`/`e5`), tokenize with padding and truncation at 512 tokens, run the encoder in `float16` (the T4 has no bf16), reduce the per-token outputs to **one vector** (`cls` = the first token's output; `mean` = the average over real tokens, ignoring padding), and **L2-normalise**, so that a dot product equals cosine similarity.

**`Reranker`: (ticket, candidate) -> relevance score**

A *bi-encoder* (the embedder above) encodes query and document separately, which is fast but coarse. A **cross-encoder** reads the query and a candidate *together*, so it can compare word-by-word and is more accurate, but it is far slower because it must run once per candidate. The standard recipe is therefore "retrieve cheaply, rerank a short list": here the top `K_CAND` = 20 candidates are re-scored and the best `K_CTX` = 3 survive.

**Memory discipline.** Both classes have a `close()` that frees their GPU storage through `release_model` plus `free_gpu`, so each phase can hand the GPU back before the next one starts. They sit on `cuda:0`; they are small (tens to a few hundred MB), unlike the 7B model.

In [9]:
import torch

class DenseEncoder:
    def __init__(self, spec, device="cuda:0"):
        from transformers import AutoTokenizer, AutoModel
        self.spec, self.device = spec, device
        self.tok = AutoTokenizer.from_pretrained(spec["id"])
        self.model = AutoModel.from_pretrained(spec["id"], dtype=torch.float16).to(device).eval()

    def encode(self, texts, kind, batch=64, max_len=512):
        prefix = self.spec["q_prefix"] if kind == "query" else self.spec["p_prefix"]
        # OPTIMIZED: sort by length so every batch pads to a similar length (less wasted GPU work); order is restored below
        order = np.argsort([len(t) for t in texts], kind="stable")
        texts = [texts[j] for j in order]
        out = []
        with torch.inference_mode():
            for i in range(0, len(texts), batch):
                enc = self.tok([prefix + t for t in texts[i:i + batch]], padding=True, truncation=True,
                               max_length=max_len, return_tensors="pt").to(self.device)
                h = self.model(**enc).last_hidden_state                   # (batch, tokens, dim)
                if self.spec["pool"] == "cls":
                    v = h[:, 0]                                           # first-token vector
                else:
                    m = enc["attention_mask"].unsqueeze(-1).to(h.dtype)   # average only over real (non-padding) tokens
                    v = (h * m).sum(1) / m.sum(1)
                out.append(torch.nn.functional.normalize(v.float(), dim=-1).cpu().numpy())
        vecs = np.vstack(out)
        res = np.empty_like(vecs); res[order] = vecs      # put every vector back at the position of its original text
        return res

    def close(self):
        release_model(self.model); del self.model; free_gpu("encoder released: ")

class Reranker:
    def __init__(self, model_id, device="cuda:0"):
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        self.device = device
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_id, dtype=torch.float16).to(device).eval()

    def score(self, query, docs, batch=16, max_len=512):
        out = []
        with torch.inference_mode():
            for i in range(0, len(docs), batch):
                enc = self.tok([query] * len(docs[i:i + batch]), docs[i:i + batch], padding=True, truncation=True,
                               max_length=max_len, return_tensors="pt").to(self.device)
                out.append(self.model(**enc).logits[:, 0].float().cpu().numpy())   # one relevance logit per pair
        return np.concatenate(out)

    def close(self):
        release_model(self.model); del self.model; free_gpu("reranker released: ")

## 2 · Retrieval experiments and the three documented decisions

### Step 2c · GPU phase 1a: embed everything once (then give the GPU back)

**What the next cell does**

It computes the vectors for *every* combination we want to compare, in one short GPU session, and then **frees the GPU**. All later comparisons are pure NumPy on the cached vectors, so trying another chunking or retriever costs milliseconds, not another model load.

**Step by step**

1. **Pre-flight** (`check_gpu_free`): these encoders need little memory, but the check protects against a forgotten kernel.
2. **Chunk the KB three ways** (`doc`, `struct`, `fixed`) and print the number of chunks each strategy produces. `struct` is exactly twice the number of documents; `fixed` shows how many tickets are long enough to be split.
3. **Prepare the query texts** for the validation and test tickets (the ticket formatted like the user message).
4. **For each embedding model, one at a time:** load it, embed the chunks of all three strategies as *passages* and the val/test tickets as *queries*, then `close()` it, which frees its GPU memory before the next model loads. The `try/finally` guarantees the release even if something fails.
5. Print one line per model: vector size, number of vectors, and seconds (a first idea of the cost differences between `small` and `base` models).

**What to look for in the output:** the chunk counts, the three embedding sessions, and GPU memory back near 0 GiB after each `encoder released` line.

In [10]:
check_gpu_free(min_free_gib=3)                       # pre-flight for GPU phase 1

CHUNKINGS = ["doc", "struct", "fixed"]
CHUNKS = {c: build_chunks(kb, c) for c in CHUNKINGS}                   # strategy -> (chunk texts, owner index)
print({c: len(CHUNKS[c][0]) for c in CHUNKINGS}, f"chunks for {len(kb)} documents")
q_texts = {"val": [ticket_text(r) for r in val_rows], "test": [ticket_text(r) for r in test_rows]}

EMB, EMB_SEC = {}, {}                                # (embedder, "chunks", chunking) / (embedder, "q", split) -> matrix
for name, spec in EMBEDDERS.items():
    t0 = time.perf_counter()
    enc = DenseEncoder(spec)
    try:                                             # always release the GPU, even if encoding fails
        for c in CHUNKINGS:
            EMB[(name, "chunks", c)] = enc.encode(CHUNKS[c][0], "passage")
        for s in q_texts:
            EMB[(name, "q", s)] = enc.encode(q_texts[s], "query")
    finally:
        enc.close()
    EMB_SEC[name] = time.perf_counter() - t0
    print(f"{name:10s} dim {EMB[(name, 'chunks', 'doc')].shape[1]} | "
          f"{sum(len(CHUNKS[c][0]) for c in CHUNKINGS) + sum(len(v) for v in q_texts.values())} vectors | {EMB_SEC[name]:.1f}s")

pre-flight: GPU 0: 0.4/44.4 GiB used | this kernel: 0.0 GiB allocated, 0.0 GiB reserved
{'doc': 800, 'struct': 1600, 'fixed': 2415} chunks for 800 documents


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

encoder released: GPU 0: 0.5/44.4 GiB used | this kernel: 0.0 GiB allocated, 0.0 GiB reserved
bge-small  dim 384 | 5095 vectors | 6.7s


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

encoder released: GPU 0: 0.5/44.4 GiB used | this kernel: 0.0 GiB allocated, 0.0 GiB reserved
bge-base   dim 768 | 5095 vectors | 3.2s


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

encoder released: GPU 0: 0.5/44.4 GiB used | this kernel: 0.0 GiB allocated, 0.0 GiB reserved
e5-base    dim 768 | 5095 vectors | 3.1s


### Step 2d · Decisions 1 and 2: chunking strategy and embedding model (with hybrid search)

**What the next cell does**

It scores **every** (chunking × retriever) combination on the **validation** tickets and applies the decision rule. It uses only the cached vectors, so it runs in seconds on the CPU.

**The retrievers compared**

`bm25` (lexical), each dense embedder, and `hybrid:<embedder>` = reciprocal-rank fusion of that embedder's ranking with BM25. Hybrid search is popular because the two methods fail differently: BM25 is exact on error codes and product names but misses paraphrases, dense vectors capture meaning but can blur rare identifiers.

**How a retrieval run works**

1. `doc_score_matrix(chunking, retriever, split)` scores every query against every chunk (BM25 scores, or the dot product of normalised vectors = cosine similarity), then collapses chunk scores to **document** scores with `to_doc_scores` (best chunk wins). Hybrid fuses two document-score matrices with `rrf_fuse`.
2. `topk` takes the best `K_CTX` = 3 documents per query, and `retrieval_metrics` scores them with the proxy defined in the design section.

**The decision rule (documented, not eyeballed)**

1. Find the best `ctx_score` in the grid.
2. Every configuration within `TOL` (0.005) of that best score is considered *tied*. A difference that small is below the noise of 100 validation tickets.
3. Among the tied configurations take the **simplest**: cheaper retriever first (`bm25` < `bge-small` < `bge-base` < `e5-base` < hybrids), then simpler chunking (`doc` < `struct` < `fixed`).
4. `CHUNKING_OVERRIDE` / `RETRIEVER_OVERRIDE` replace the automatic choice if you have a reason.

**What the cell prints**

- the **pivot table** (retrievers × chunking) of `ctx_score`, coloured; this is the main evidence for the report;
- the **full table** with all five proxy metrics, sorted by `ctx_score`;
- two lines starting with `DECISION`, which you can paste into the report.

**How to read it:** look at the *spread*. If all cells are within a few thousandths, the choice barely matters and the cheap option is justified. A large gap between `bm25` and the dense models would say that semantic matching adds real value on this data.

In [11]:
from IPython.display import display

RETRIEVERS = ["bm25"] + list(EMBEDDERS) + [f"hybrid:{e}" for e in EMBEDDERS]      # order = from cheapest to most complex
BM25S = {c: BM25(CHUNKS[c][0]) for c in CHUNKINGS}                                  # one lexical index per chunking
_bm_cache = {}

def doc_score_matrix(chunking, retriever, split):
    """(n_queries x n_docs) document scores for one configuration."""
    _, owner = CHUNKS[chunking]; n = len(kb)
    def bm():
        if (chunking, split) not in _bm_cache:
            _bm_cache[(chunking, split)] = to_doc_scores(BM25S[chunking].scores(q_texts[split]), owner, n)
        return _bm_cache[(chunking, split)]
    def dense(e):
        return to_doc_scores(EMB[(e, "q", split)] @ EMB[(e, "chunks", chunking)].T, owner, n)   # cosine similarity
    if retriever == "bm25":
        return bm()
    if retriever.startswith("hybrid:"):
        return rrf_fuse([dense(retriever.split(":")[1]), bm()], RRF_K)
    return dense(retriever)

rows = []
for c in CHUNKINGS:
    for r in RETRIEVERS:
        top = topk(doc_score_matrix(c, r, "val"), K_CTX)
        rows.append({"chunking": c, "retriever": r, **retrieval_metrics(top, val_rows, kb)})
grid = pd.DataFrame(rows)

# decision rule: best score, ties (within TOL) resolved in favour of the simpler / cheaper option
best = grid["ctx_score"].max()
tied = grid[grid["ctx_score"] >= best - TOL].copy()
tied["_order"] = [RETRIEVERS.index(r) * 10 + CHUNKINGS.index(c) for c, r in zip(tied["chunking"], tied["retriever"])]
pick = tied.sort_values("_order").iloc[0]
CHUNKING = CHUNKING_OVERRIDE or pick["chunking"]
RETRIEVER = RETRIEVER_OVERRIDE or pick["retriever"]

pivot = grid.pivot(index="retriever", columns="chunking", values="ctx_score").loc[RETRIEVERS, CHUNKINGS]
print("ctx_score on the VALIDATION tickets (higher = retrieved tickets are more useful):")
display(pivot.style.format("{:.3f}").background_gradient(cmap="Greens", axis=None))
display(grid.sort_values("ctx_score", ascending=False).style.format({c: "{:.3f}" for c in grid.columns[2:]}))
print(f"{len(tied)} of {len(grid)} configurations are within TOL={TOL} of the best ({best:.3f}).")
print(f"DECISION 1 (chunking):  '{CHUNKING}'  |  DECISION 2 (retriever): '{RETRIEVER}'"
      + ("  [override]" if CHUNKING_OVERRIDE or RETRIEVER_OVERRIDE else ""))

ctx_score on the VALIDATION tickets (higher = retrieved tickets are more useful):


chunking,doc,struct,fixed
retriever,,,
bm25,0.500,0.509,0.493
bge-small,0.479,0.493,0.476
bge-base,0.475,0.480,0.479
e5-base,0.479,0.486,0.479
hybrid:bge-small,0.491,0.507,0.499
hybrid:bge-base,0.487,0.510,0.501
hybrid:e5-base,0.509,0.496,0.502


,chunking,retriever,queue_p,type_p,priority_p,res_rougeL,ctx_score
12,struct,hybrid:bge-base,0.573,0.683,0.533,0.249,0.510
6,doc,hybrid:e5-base,0.580,0.687,0.520,0.250,0.509
7,struct,bm25,0.573,0.677,0.533,0.251,0.509
11,struct,hybrid:bge-small,0.563,0.670,0.543,0.250,0.507
20,fixed,hybrid:e5-base,0.540,0.707,0.507,0.254,0.502
19,fixed,hybrid:bge-base,0.550,0.677,0.527,0.249,0.501
0,doc,bm25,0.567,0.663,0.523,0.246,0.500
18,fixed,hybrid:bge-small,0.547,0.677,0.523,0.248,0.499
13,struct,hybrid:e5-base,0.553,0.690,0.490,0.251,0.496
14,fixed,bm25,0.547,0.647,0.537,0.242,0.493


4 of 21 configurations are within TOL=0.005 of the best (0.510).
DECISION 1 (chunking):  'struct'  |  DECISION 2 (retriever): 'bm25'


### Step 2e · Decision 3: is a reranker worth its latency?

**What the next cell does**

It loads the cross-encoder, re-scores the top `K_CAND` candidates of the chosen retriever for every validation ticket, and compares the final top-3 **with** and **without** reranking using the same proxy metrics. Then it applies the decision rule.

**Step by step**

1. **Pre-flight**, then load `bge-reranker-base` (it stays loaded for the next cell).
2. `rerank_top(rows, S, k_cand, k_out)`: for each ticket take the `K_CAND` best candidates from the retriever's score matrix `S`, build one text per candidate (the *whole* parent ticket), let the cross-encoder score every `(ticket, candidate)` pair, and keep the `k_out` best.
3. Compute `retrieval_metrics` for the plain top-3 and for the reranked top-3, and the **gain** in `ctx_score`.
4. **Measure the cost:** seconds per query spent in the reranker (GPU-synchronised timing).
5. **Rule:** adopt the reranker only if `gain >= RERANK_MIN_GAIN` (0.01); `RERANK_OVERRIDE` forces True/False.

**Why this rule:** a cross-encoder adds a model, GPU memory and latency to every request. Unless it measurably improves what lands in the prompt, it is not worth it, and "no reranker" is a perfectly good, simpler answer for the report.

**What to look for in the output:** the two-row comparison table, the gain, the added milliseconds per query, and the `DECISION 3` line.

In [12]:
check_gpu_free(min_free_gib=3)
reranker = Reranker(RERANKER_ID)                      # stays loaded for the final retrieval; closed at the end of that cell

def rerank_top(rows, S, k_cand, k_out):
    """For each query: take the k_cand best candidates of score matrix S, re-score them with the cross-encoder, keep k_out."""
    cand = topk(S, k_cand); out = []
    for qi, q in enumerate(rows):
        # the cross-encoder reads (ticket, candidate) together: slower than the bi-encoder, so it only sees the short list
        sc = reranker.score(ticket_text(q), [full_text(kb[j]) for j in cand[qi]])   # one logit per (ticket, candidate)
        out.append(cand[qi][np.argsort(-sc)[:k_out]])
    return np.array(out)

S_val = doc_score_matrix(CHUNKING, RETRIEVER, "val")
plain_top = topk(S_val, K_CTX)
sync(); t0 = time.perf_counter()
rr_top = rerank_top(val_rows, S_val, K_CAND, K_CTX)
sync(); rerank_ms = (time.perf_counter() - t0) / len(val_rows) * 1000

m_plain, m_rr = retrieval_metrics(plain_top, val_rows, kb), retrieval_metrics(rr_top, val_rows, kb)
cmp_tbl = pd.DataFrame({"no reranker": m_plain, "with reranker": m_rr}).T
cmp_tbl.loc["gain"] = cmp_tbl.loc["with reranker"] - cmp_tbl.loc["no reranker"]
display(cmp_tbl.style.format("{:+.3f}", subset=pd.IndexSlice[["gain"], :]).format("{:.3f}", subset=pd.IndexSlice[["no reranker", "with reranker"], :]))

gain = m_rr["ctx_score"] - m_plain["ctx_score"]
USE_RERANK = bool(RERANK_OVERRIDE) if RERANK_OVERRIDE is not None else gain >= RERANK_MIN_GAIN
print(f"reranker gain in ctx_score: {gain:+.3f} (rule: adopt if >= {RERANK_MIN_GAIN}) | extra cost: {rerank_ms:.0f} ms/query")
print(f"DECISION 3 (reranker): {'USE ' + RERANKER_ID if USE_RERANK else 'do not use a reranker'}"
      + ("  [override]" if RERANK_OVERRIDE is not None else ""))

pre-flight: GPU 0: 0.5/44.4 GiB used | this kernel: 0.0 GiB allocated, 0.0 GiB reserved


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,queue_p,type_p,priority_p,res_rougeL,ctx_score
no reranker,0.573,0.677,0.533,0.251,0.509
with reranker,0.520,0.640,0.457,0.241,0.464
gain,-0.053,-0.037,-0.077,-0.010,-0.044


reranker gain in ctx_score: -0.044 (rule: adopt if >= 0.01) | extra cost: 25 ms/query
DECISION 3 (reranker): do not use a reranker


### Step 3 · Final retrieval for the validation and test tickets, latency, and freeing GPU phase 1

**What the next cell does**

It runs the **chosen** pipeline (decisions 1–3) for every validation and test ticket, saves what was retrieved, measures the real per-request latency, saves the configuration for Week 4, and then **releases the encoder and reranker** so the GPU is empty for the 7B model.

**Step by step**

1. `retrieve(split, rows)` applies the chosen chunking and retriever (and the reranker if adopted) and returns the top-`K_CTX` documents per ticket. The result, `RET[split]`, stores only the *ids and scores* of the documents, so it is small and can be inspected or re-used by the serving code.
2. **Descriptive retrieval quality on test.** The proxy metrics are also computed for the test tickets. They are reported for the write-up but **no decision uses them**.
3. **Latency, measured like production.** Single queries, one after another, with two warm-up queries first: embed the ticket (live, on the GPU), score the KB, fuse with BM25 if hybrid, and rerank if adopted. We report the median (p50) and the 95th percentile (p95) in milliseconds. This is the retrieval overhead every RAG request pays before the LLM even starts.
4. **Save the decisions.** `retrieval_decisions.json` holds the complete grid, the rerank comparison and the rules, so the decisions are reproducible and citable. `rag_config.json` holds the final pipeline settings that the Week 4 FastAPI service must implement.
5. **Release GPU phase 1:** close the encoder and the reranker, drop references, and call `free_gpu`. The cached vectors stay in RAM.

**What to look for in the output:** retrieval quality on val and test (they should be similar; a much higher test number would be suspicious), the latency percentiles, and GPU memory back near zero at the end.

In [13]:
def retrieve(split, rows):
    # Stage 1: score every KB document with the chosen retriever. Stage 2 (optional): the cross-encoder re-orders the top K_CAND.
    # NOTE: the "scores" saved in retrieved_*.json are these first-stage scores, even when a reranker picked the final order.
    S = doc_score_matrix(CHUNKING, RETRIEVER, split)
    top = rerank_top(rows, S, K_CAND, K_CTX) if USE_RERANK else topk(S, K_CTX)
    return top, S

RET, RET_METRICS = {}, {}
for split, rows_ in [("val", val_rows), ("test", test_rows)]:
    top, S = retrieve(split, rows_)
    RET[split] = [{"uid": r["uid"], "docs": [kb[j]["kb_id"] for j in t], "scores": [float(S[qi, j]) for j in t]}
                  for qi, (r, t) in enumerate(zip(rows_, top))]
    (OUT / f"retrieved_{split}.json").write_text(json.dumps(RET[split]))
    RET_METRICS[split] = retrieval_metrics(top, rows_, kb)
print(pd.DataFrame(RET_METRICS).T.round(3).to_string(), "\n(test numbers are descriptive only; no decision used them)\n")

# ---- latency of the chosen pipeline, one query at a time (like production), after two warm-up queries
emb_name = None if RETRIEVER == "bm25" else RETRIEVER.split(":")[-1]
encoder = DenseEncoder(EMBEDDERS[emb_name]) if emb_name else None
_, owner_c = CHUNKS[CHUNKING]

def one_query_ms(row):
    sync(); t0 = time.perf_counter()
    text = ticket_text(row)
    parts = []
    if RETRIEVER == "bm25" or RETRIEVER.startswith("hybrid:"):
        parts.append(to_doc_scores(BM25S[CHUNKING].scores([text]), owner_c, len(kb)))
    if emb_name:
        qv = encoder.encode([text], "query")                                          # live embedding of the ticket
        parts.insert(0, to_doc_scores(qv @ EMB[(emb_name, "chunks", CHUNKING)].T, owner_c, len(kb)))
    S1 = rrf_fuse(parts, RRF_K) if RETRIEVER.startswith("hybrid:") else parts[0]
    _ = rerank_top([row], S1, K_CAND, K_CTX) if USE_RERANK else topk(S1, K_CTX)
    sync(); return (time.perf_counter() - t0) * 1000

sample = test_rows[:min(30, len(test_rows))]
for r in sample[:2]: one_query_ms(r)                                                   # warm-up
lat = np.array([one_query_ms(r) for r in sample])
RETRIEVAL_MS = {"p50": float(np.percentile(lat, 50)), "p95": float(np.percentile(lat, 95)), "n": len(lat)}
print(f"retrieval latency (single query): p50 {RETRIEVAL_MS['p50']:.0f} ms | p95 {RETRIEVAL_MS['p95']:.0f} ms | n={len(lat)}")

# ---- save decisions and the pipeline configuration for Week 4
(OUT / "retrieval_decisions.json").write_text(json.dumps({
    "rules": {"tol": TOL, "rerank_min_gain": RERANK_MIN_GAIN, "primary_metric": "ctx_score on validation"},
    "grid": grid.to_dict("records"),
    "rerank_eval": {"no_reranker": m_plain, "with_reranker": m_rr, "gain": gain, "extra_ms_per_query": rerank_ms},
    "chosen": {"chunking": CHUNKING, "retriever": RETRIEVER, "use_reranker": USE_RERANK}}, indent=2))
RAG_CONFIG = {"kb_sha256": kb_sha, "n_docs": len(kb), "chunking": CHUNKING, "retriever": RETRIEVER,
              "embedder": EMBEDDERS.get(emb_name), "reranker": RERANKER_ID if USE_RERANK else None,
              "k_cand": K_CAND, "k_ctx": K_CTX, "rrf_k": RRF_K, "fixed_words": FIXED_WORDS, "fixed_stride": FIXED_STRIDE,
              "max_doc_chars": MAX_DOC_CHARS, "context_with_labels": CONTEXT_WITH_LABELS, "prompt_version": RAG_PROMPT_VERSION,
              "retrieval_latency_ms": RETRIEVAL_MS}
(OUT / "rag_config.json").write_text(json.dumps(RAG_CONFIG, indent=2))

# ---- release GPU phase 1 (encoder + reranker); the cached vectors stay in RAM
if encoder is not None:
    encoder.close()
reranker.close()
del encoder, reranker
free_gpu("GPU phase 1 finished: ")

      queue_p  type_p  priority_p  res_rougeL  ctx_score
val     0.573   0.677       0.533       0.251      0.509
test    0.567   0.676       0.467       0.254      0.491 
(test numbers are descriptive only; no decision used them)

retrieval latency (single query): p50 0 ms | p95 0 ms | n=30


reranker released: GPU 0: 0.5/44.4 GiB used | this kernel: 0.0 GiB allocated, 0.0 GiB reserved
GPU phase 1 finished: GPU 0: 0.5/44.4 GiB used | this kernel: 0.0 GiB allocated, 0.0 GiB reserved


## 3 · Generation with retrieval

### Step 4a · Build the RAG prompt and check how much longer it is

**What the next cell does**

It loads the tokenizer (small, no GPU), turns the retrieved ids back into full KB documents, defines how a RAG prompt is assembled, prints one complete example exactly as the model will see it, and measures how many tokens the RAG prompts add compared with the plain prompts.

**Design of the RAG prompt**

- The **system prompt is unchanged** (the same `SYSTEM` as Weeks 1–2). That matters for the fine-tuned model: it was trained with exactly this system prompt, and changing it would confound "RAG helps" with "the prompt changed".
- The **retrieved tickets go into the user message**, before the new ticket: a short instruction ("use them as guidance only; they may not match exactly"), then `[KB-1] ... [KB-3]`, each with its labels (if `CONTEXT_WITH_LABELS`), the ticket and its resolution, then `---` and the new ticket. Each retrieved text is truncated to `MAX_DOC_CHARS`.
- **A caveat for the fine-tuned model:** during fine-tuning it only ever saw *ticket in, JSON out*, never a context block. So fine-tuned + RAG tests whether it can use context it was not trained on, or whether it learned to ignore everything except the ticket. Either outcome is a legitimate finding.

**What the cell prints**

1. One full prompt (cut at 3,500 characters) in the model's chat format, so you can read what the LLM actually receives.
2. Prompt-length statistics in tokens (median, 95th percentile, maximum) for plain vs RAG prompts. RAG prompts are longer, so generation is slower; this is part of the cost/benefit analysis later. The maximum must stay far below the model's context window (32k tokens for Qwen2.5), which it does by a wide margin.

In [14]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token                    # needed so batches of different lengths can be padded
tok.padding_side = "left"                            # decoder-only models must be left-padded for batched generation

def render(msgs, add_generation_prompt=False):
    """Chat-template text; same function is used for training and inference."""
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=add_generation_prompt)
    except Exception:                                # templates without a system role
        # fallback: glue the system text and the first user message into a single user message
        merged = [{"role": "user", "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]}] + msgs[2:]
        return tok.apply_chat_template(merged, tokenize=False, add_generation_prompt=add_generation_prompt)

kb_by_id = {x["kb_id"]: x for x in kb}
# retrieved ids -> full KB documents, per split and ticket
RET_DOCS = {s: {r["uid"]: [kb_by_id[i] for i in r["docs"]] for r in RET[s]} for s in RET}

def rag_msgs(row, docs):
    # SAME system prompt as Weeks 1-2; the retrieved tickets travel in the user message
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": rag_user_message(row, docs)}]

ex = test_rows[0]
print(render(rag_msgs(ex, RET_DOCS["test"][ex["uid"]]), True)[:3500], "\n...\n")

ntok = lambda text: len(tok(text, add_special_tokens=False)["input_ids"])
L_plain = np.array([ntok(render(r["messages"][:2], True)) for r in test_rows])
L_rag = np.array([ntok(render(rag_msgs(r, RET_DOCS["test"][r["uid"]]), True)) for r in test_rows])
print(pd.DataFrame({"plain prompt": L_plain, "RAG prompt": L_rag}).describe(percentiles=[.5, .95]).loc[["50%", "95%", "max"]].round(0).to_string())

<|im_start|>system
You are an enterprise IT helpdesk assistant. Read the support ticket and answer with exactly one JSON object and no other text.
Keys:
- "queue": one of ["Technical Support", "IT Support", "Service Outages and Maintenance"]
- "type": one of ["Change", "Incident", "Problem", "Request"]
- "priority": one of ["high", "low", "medium"]
- "resolution": a concise, professional reply to the user with concrete next steps (under 150 words). Never ask for passwords and never invent policy details.<|im_end|>
<|im_start|>user
Resolved tickets from the knowledge base (use them as guidance only; they may not match this ticket exactly):

[KB-1] Queue: Technical Support | Type: Incident | Priority: high
Ticket: Technical Support for Connectivity Problems - Users are facing connectivity challenges across various devices and applications, which is disrupting project workflows. These issues might be due to recent software updates or network changes. We have attempted to resolve by restar

     plain prompt  RAG prompt
50%         203.0       736.0
95%         252.0       927.0
max         316.0      1066.0


### Step 4b · GPU phase 2: load the 7B base model

**What the next cell does**

It starts the second GPU phase. The small encoders are gone (phase 1 released them), so the GPU is free for the 7B language model.

**Step by step**

1. **Pre-flight** with `check_gpu_free(min_free_gib=10)`: the 4-bit weights need about 5.5 GB plus room for the longer RAG prompts. If another kernel holds the GPU, we stop here with the fix instead of crashing after a slow load.
2. The cell defines `load_base()`, `msgs_for()` and `generate_batch()`: **exact copies** of the Week 2 functions. `load_base` loads the model **the same way as Weeks 1–2** (4-bit NF4 with double quantization, compute precision `float16` on a T4, `device_map="auto"`) and refuses to run if the precision differs from the one recorded in the Week 1 baseline. `generate_batch` pads a batch of prompts together, generates with **greedy decoding** (`do_sample=False`, so results are repeatable), times the GPU work with `synchronize`, slices off the prompt, and counts the new tokens.
3. Load the model and print a one-line summary.

**Why "identical to Week 2" matters:** the comparison "fine-tuned vs fine-tuned + RAG" is only meaningful if *nothing but the prompt* differs: same quantization, same decoding, same token limit, same batch size.

**What to look for in the output:** the model name, the GPU name and `float16`; afterwards roughly 5–6 GB spread over the GPUs.

In [15]:
check_gpu_free(min_free_gib=10)                       # pre-flight for GPU phase 2 (the 7B model)

def seed_everything(s):
    # one call seeds every random generator involved, so a run can be repeated exactly
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def load_base():
    # Same loading recipe as Week 1: 4-bit NF4 base + explicit compute dtype -> a fair base-vs-fine-tuned comparison
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    # bfloat16 only on GPUs with native support (Ampere+); a T4 only emulates it (~80 s per step measured), so use float16
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
    if BASELINE_DTYPE and str(dtype) != BASELINE_DTYPE:   # same precision as the baseline, or the comparison is unfair
        raise RuntimeError(f"Week 1 baseline used {BASELINE_DTYPE} but this GPU selects {dtype}. Re-run Week 1 on this machine.")
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", quantization_config=q, dtype=dtype), dtype

def msgs_for(r):
    return r["messages"][:2]                          # system + user (zero-shot format used in training)

def generate_batch(model, msgs_list):
    # pad the prompts into one batch so the GPU handles them together
    enc = tok([render(m, True) for m in msgs_list], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    torch.cuda.synchronize(); t0 = time.perf_counter()     # synchronize: GPU work is async, so wait before reading the clock
    with torch.inference_mode():
        # greedy decoding (do_sample=False) -> deterministic output; None values silence unused-sampling warnings
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None,
                             top_k=None, pad_token_id=tok.pad_token_id, use_cache=True)
    torch.cuda.synchronize(); dt = time.perf_counter() - t0
    gen = out[:, enc["input_ids"].shape[1]:]          # generate() returns prompt + answer; keep only the new tokens
    n_tok = [int((g != tok.pad_token_id).sum()) for g in gen]
    return tok.batch_decode(gen, skip_special_tokens=True), n_tok, dt, int(enc["input_ids"].shape[1])

model, dtype = load_base()
model.eval()
print(MODEL_ID, "| 4-bit |", dtype, "| GPU:", torch.cuda.get_device_name(0))
gpu_report("model loaded: ")

pre-flight: GPU 0: 0.5/44.4 GiB used | this kernel: 0.0 GiB allocated, 0.0 GiB reserved


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen/Qwen2.5-7B-Instruct | 4-bit | torch.bfloat16 | GPU: NVIDIA L40S
model loaded: GPU 0: 5.8/44.4 GiB used | this kernel: 5.2 GiB allocated, 5.3 GiB reserved


### Step 4c · Generate: base + RAG, then fine-tuned + RAG

**What the next cell does**

It answers every test ticket with each RAG system and saves the raw outputs. This is the longest-running cell of the notebook (the RAG prompts are 3–4 times longer than the plain ones).

**Step by step**

1. `run_system(...)` goes through the test tickets in batches of `GEN_BATCH`, builds either the RAG prompt or the plain prompt, calls `generate_batch`, and records for each ticket the text, token count, batch time, batch size and prompt length (so speed can be computed exactly as in Weeks 1–2).
2. **The jobs, in a deliberate order.** `base_rag` first, with the plain base model. Only then is the Week 2 adapter attached with `PeftModel.from_pretrained` (the `ft_model`). Attaching an adapter modifies the base model's layers in place, so any later *base* job uses `ft_model.disable_adapter()`: the code is safe to re-run. If `REGEN_NO_RAG = True` the two no-RAG systems are regenerated here too.
3. **Resume with a safety check.** Each job's predictions go to `preds_<job>.jsonl` next to a `.meta.json` **fingerprint** of everything that influences the text: model, precision, decoding settings, batch size, system prompt, test-set hash, the adapter file's hash, the KB hash, the retrieval decisions and the prompt version. Saved predictions are re-used only if the fingerprint matches; otherwise they are regenerated, so stale answers can never leak into the results.
4. When done, `free_gpu` returns the memory PyTorch cached for activations and the key-value cache. The model stays loaded until the cleanup cell.

**What to look for in the output:** progress bars for each job, one line per job with the file and row count (should equal the test size), and GPU memory afterwards.

In [16]:
from tqdm.auto import tqdm
from contextlib import nullcontext
from peft import PeftModel

def run_system(target, rows, split, use_rag, desc):
    recs = []
    for i in tqdm(range(0, len(rows), GEN_BATCH), desc=desc):
        b = rows[i:i + GEN_BATCH]
        msgs = [rag_msgs(r, RET_DOCS[split][r["uid"]]) if use_rag else msgs_for(r) for r in b]   # RAG prompt or plain prompt
        texts, n_tok, dt, plen = generate_batch(target, msgs)
        recs += [{"uid": r["uid"], "pred_text": t, "gen_tokens": n, "batch_sec": dt, "batch_size": len(b), "prompt_len": plen}
                 for r, t, n in zip(b, texts, n_tok)]
    return pd.DataFrame(recs)

# fingerprint of everything that changes the generated text (saved predictions are reused ONLY if it matches)
sha16 = lambda s: hashlib.sha256(s.encode()).hexdigest()[:16]
weights = next((ADAPTER_DIR / f for f in ["adapter_model.safetensors", "adapter_model.bin"] if (ADAPTER_DIR / f).exists()), None)
fp_common = {"model": MODEL_ID, "dtype": str(dtype), "max_new_tokens": MAX_NEW_TOKENS, "gen_batch": GEN_BATCH, "smoke": SMOKE,
             "n_test": len(test_rows), "system_sha": sha16(SYSTEM), "test_sha": man["sha256"]["test"][:16]}
fp_rag = {**fp_common, "kb_sha": kb_sha[:16], "chunking": CHUNKING, "retriever": RETRIEVER, "rerank": USE_RERANK,
          "k_cand": K_CAND, "k_ctx": K_CTX, "prompt_version": RAG_PROMPT_VERSION, "labels_in_ctx": CONTEXT_WITH_LABELS,
          "max_doc_chars": MAX_DOC_CHARS}
adapter_sha = hashlib.sha256(weights.read_bytes()).hexdigest()[:16] if weights else "none"

# (tag, which model, use retrieval). Base jobs come before the adapter is attached.
JOBS = [("base_rag", "base", True)] + ([("base_norag", "base", False)] if REGEN_NO_RAG else []) \
     + [("ft_rag", "ft", True)] + ([("ft_norag", "ft", False)] if REGEN_NO_RAG else [])
ft_model = globals().get("ft_model")                 # survives a re-run of this cell
preds = {}
for tag, kind, use_rag in JOBS:
    p = OUT / f"preds_{tag}{'_smoke' if SMOKE else ''}.jsonl"
    meta = p.with_suffix(".meta.json")
    fp = {**(fp_rag if use_rag else fp_common), "kind": kind, "adapter_sha": adapter_sha if kind == "ft" else None}
    if p.exists() and meta.exists() and json.loads(meta.read_text()) == fp:
        preds[tag] = pd.read_json(p, lines=True)
    else:
        if p.exists():
            print(f"{p.name}: saved with different (or unknown) settings -> regenerating")
        if kind == "ft" and ft_model is None:
            ft_model = PeftModel.from_pretrained(model, ADAPTER_DIR).eval()      # attach the Week 2 LoRA adapter
        target = ft_model if ft_model is not None else model
        ctx = ft_model.disable_adapter() if (kind == "base" and ft_model is not None) else nullcontext()
        with ctx:                                                                # base jobs: adapter switched off
            preds[tag] = run_system(target, test_rows, "test", use_rag, tag)
        preds[tag].to_json(p, orient="records", lines=True, force_ascii=False)
        meta.write_text(json.dumps(fp, indent=2))                                # fingerprint last: half-written files are never trusted
    print(tag, "->", p, len(preds[tag]), "rows")
free_gpu("generation done: ")

base_rag -> capstone_it_helpdesk/week3/preds_base_rag.jsonl 180 rows
ft_rag -> capstone_it_helpdesk/week3/preds_ft_rag.jsonl 180 rows


generation done: GPU 0: 5.7/44.4 GiB used | this kernel: 5.2 GiB allocated, 5.2 GiB reserved


## 4 · Evaluation

### Step 5a · Score all five systems with the same ruler and build the comparison tables

**What the next cell does**

It collects the predictions of all five systems, scores every one of them with the identical Week 1/2 scoring code, and prints the central results tables.

**Where each system's answers come from**

- *base · zero-shot*: Week 1's `preds_zero_shot.jsonl`.
- *fine-tuned*: Week 2's `preds_<FT_RUN>.jsonl` (or this notebook's regenerated file if `REGEN_NO_RAG`).
- *base + RAG*, *fine-tuned + RAG*: generated in Step 4c.
- *retrieval-only (kNN)*: built on the spot, **no LLM**: majority vote of the three retrieved tickets' `queue`/`type`/`priority`, and the top-1 retrieved ticket's reply as the `resolution`. It is always schema-valid by construction, costs only the retrieval time, and answers an important question: *how much of the result is just "find a similar ticket", and what does the LLM add on top?*
- *majority-class reference*: a table row, not a model: always answer the most common **training** label. It is the floor every label metric must clear (the Week 1 analysis showed the zero-shot model barely does).

**How scoring works**

`score_system` aligns predictions with the gold tickets by `uid`, runs `score_row` on every ticket and `summarize` on the result, and computes the speed numbers: every row carries its batch's wall time, so dividing by the batch size and summing counts each batch once. The per-ticket scores are kept (`SCORED`) for the statistics and the error analysis.

**The tables**

1. **Quality table**: schema validity, accuracies, labels-exact-match, queue/priority macro-F1, ROUGE-L, with a green gradient.
2. **Cost table**: mean prompt length, generation speed (tokens/s), generation seconds per ticket, retrieval latency (p50, ms), and **end-to-end seconds per ticket** = generation + retrieval. This is the price of each quality gain.

**How to read the quality table:** compare systems *in pairs*, each pair isolating one change: 1→2 is the effect of fine-tuning, 1→3 of retrieval on the base model, 2→4 of retrieval on the fine-tuned model, 3→4 of fine-tuning once retrieval is present. With 180 test tickets, one ticket is 0.56 percentage points; the next cell attaches confidence intervals, and you should not read much into differences of 1–3 points before checking them.

In [17]:
from IPython.display import display
from sklearn.metrics import f1_score

uids = [r["uid"] for r in test_rows]
gold = pd.DataFrame(test_rows)

def load_preds(path):
    # align by ticket id (a shuffled or longer file can never pair a reply with the wrong ticket)
    return pd.read_json(path, lines=True).set_index("uid").loc[uids].reset_index()

def score_system(preds):
    p = preds.set_index("uid").loc[uids].reset_index()
    sc = pd.DataFrame([score_row(t, g, QUEUES, TYPES, PRIORITIES) for t, g in zip(p["pred_text"], gold.to_dict("records"))])
    m = summarize(sc, gold, QUEUES, PRIORITIES)
    tot = (p["batch_sec"] / p["batch_size"]).sum()          # each batch's wall time counted once
    m.update({"mean_prompt_len": p["prompt_len"].mean(), "mean_out_tokens": p["gen_tokens"].mean(),
              "gen_sec_per_ticket": tot / len(p), "tokens_per_sec": p["gen_tokens"].sum() / tot if tot > 0 else np.nan})
    return {k: float(v) for k, v in m.items()}, pd.concat([p.drop(columns=[c for c in sc.columns if c in p.columns]), sc], axis=1)

# retrieval-only system: vote of the retrieved labels + the top-1 retrieved reply (no LLM, so no generation time)
knn = pd.DataFrame([{"uid": u, "pred_text": knn_prediction(RET_DOCS["test"][u]), "gen_tokens": 0, "batch_sec": 0.0,
                     "batch_size": 1, "prompt_len": 0} for u in uids])

N_BASE, N_FT, N_BRAG, N_FRAG, N_KNN = "base · zero-shot", "fine-tuned", "base + RAG", "fine-tuned + RAG", "retrieval-only (kNN)"
SYS = {
    N_BASE: preds["base_norag"] if "base_norag" in preds else load_preds(W1 / "preds_zero_shot.jsonl"),
    N_FT:   preds["ft_norag"] if "ft_norag" in preds else load_preds(W2 / f"preds_{FT_RUN}.jsonl"),
    N_BRAG: preds["base_rag"], N_FRAG: preds["ft_rag"], N_KNN: knn}
RETRIEVAL_COST = {N_BASE: 0.0, N_FT: 0.0, N_BRAG: RETRIEVAL_MS["p50"], N_FRAG: RETRIEVAL_MS["p50"], N_KNN: RETRIEVAL_MS["p50"]}

METRICS, SCORED = {}, {}
for name, df_ in SYS.items():
    METRICS[name], SCORED[name] = score_system(df_)
    METRICS[name]["retrieval_ms_p50"] = RETRIEVAL_COST[name]
    METRICS[name]["e2e_sec_per_ticket"] = METRICS[name]["gen_sec_per_ticket"] + RETRIEVAL_COST[name] / 1000

# majority-class floor, using TRAIN labels only (the test set is never peeked at)
maj = {k: pd.Series([r[k] for r in train_rows]).mode()[0] for k in ["queue", "type", "priority"]}
floor = {"queue_acc": (gold["queue"] == maj["queue"]).mean(), "type_acc": (gold["type"] == maj["type"]).mean(),
         "priority_acc": (gold["priority"] == maj["priority"]).mean(),
         "labels_exact_match": ((gold["queue"] == maj["queue"]) & (gold["type"] == maj["type"]) & (gold["priority"] == maj["priority"])).mean(),
         "queue_macro_f1": f1_score(gold["queue"], [maj["queue"]] * len(gold), labels=QUEUES, average="macro", zero_division=0),
         "priority_macro_f1": f1_score(gold["priority"], [maj["priority"]] * len(gold), labels=PRIORITIES, average="macro", zero_division=0)}

res = pd.DataFrame(METRICS).T
res.loc["majority-class floor"] = pd.Series(floor)
pct = ["json_parse_rate", "schema_valid_rate", "queue_acc", "type_acc", "priority_acc", "labels_exact_match"]
KEYM = ["schema_valid_rate", "labels_exact_match", "queue_macro_f1", "priority_macro_f1", "rougeL_resolution"]
print(f"QUALITY on the {len(gold)} held-out test tickets (the majority-class row is a floor, not a model):")
display(res[pct + ["queue_macro_f1", "priority_macro_f1", "rougeL_resolution"]].style
        .format({**{c: "{:.1%}" for c in pct}, "queue_macro_f1": "{:.3f}", "priority_macro_f1": "{:.3f}", "rougeL_resolution": "{:.3f}"}, na_rep="-")
        .background_gradient(cmap="Greens", subset=pd.IndexSlice[list(SYS), pct[1:] + ["queue_macro_f1", "priority_macro_f1", "rougeL_resolution"]]))
print("\nCOST per ticket (HF generate, 4-bit, LoRA unmerged; final serving numbers come from vLLM in Week 4):")
cost_cols = ["mean_prompt_len", "mean_out_tokens", "tokens_per_sec", "gen_sec_per_ticket", "retrieval_ms_p50", "e2e_sec_per_ticket"]
display(res.loc[list(SYS), cost_cols].style.format({"mean_prompt_len": "{:.0f}", "mean_out_tokens": "{:.0f}", "tokens_per_sec": "{:.1f}",
        "gen_sec_per_ticket": "{:.2f}", "retrieval_ms_p50": "{:.0f}", "e2e_sec_per_ticket": "{:.2f}"}, na_rep="-"))

QUALITY on the 180 held-out test tickets (the majority-class row is a floor, not a model):


,json_parse_rate,schema_valid_rate,queue_acc,type_acc,priority_acc,labels_exact_match,queue_macro_f1,priority_macro_f1,rougeL_resolution
base · zero-shot,100.0%,100.0%,65.0%,72.8%,50.0%,20.0%,0.494,0.379,0.220
fine-tuned,100.0%,100.0%,67.2%,78.3%,53.3%,28.9%,0.475,0.242,0.312
base + RAG,100.0%,100.0%,62.8%,71.1%,56.7%,26.7%,0.510,0.398,0.322
fine-tuned + RAG,100.0%,100.0%,62.8%,78.3%,55.0%,27.2%,0.469,0.389,0.345
retrieval-only (kNN),100.0%,100.0%,59.4%,71.7%,51.1%,21.1%,0.455,0.397,0.295
majority-class floor,-,-,64.4%,50.0%,53.9%,20.0%,0.261,0.233,-



COST per ticket (HF generate, 4-bit, LoRA unmerged; final serving numbers come from vLLM in Week 4):


,mean_prompt_len,mean_out_tokens,tokens_per_sec,gen_sec_per_ticket,retrieval_ms_p50,e2e_sec_per_ticket
base · zero-shot,254,91,270.2,0.34,0,0.34
fine-tuned,254,92,107.6,0.85,0,0.85
base + RAG,914,107,224.4,0.48,0,0.48
fine-tuned + RAG,914,109,108.3,1.00,0,1.00
retrieval-only (kNN),0,0,-,0.00,0,0.00


### Step 5b · Is the difference real? Paired bootstrap confidence intervals, and does the model use the context?

**What the next cell does**

Two things that turn a table of numbers into evidence.

**1. Paired bootstrap confidence intervals**

With only 180 test tickets, chance alone moves a score by several points, so "system A scored 2 points higher than B" is not yet a finding. The **paired bootstrap** quantifies the uncertainty:

1. For every ticket we have a per-ticket score for both systems (for example `1` if all three labels are right, `0` otherwise, or the ROUGE-L of its reply). The *difference* per ticket is `A − B`.
2. Resample the 180 tickets **with replacement** `N_BOOT` times (the *same* resampled tickets for both systems, which is what "paired" means and why it is more sensitive than comparing two separate error bars), and average the differences each time.
3. The 2.5th and 97.5th percentiles of those averages form a **95% confidence interval**. If it **excludes 0**, the difference is unlikely to be chance (marked `clear`); if it includes 0, the data cannot distinguish the systems (marked `within noise`).

The four comparisons are the four questions of the study: fine-tuning on top of RAG (`FT+RAG` vs `FT`... i.e. what RAG adds to the fine-tuned model), what RAG adds to the base model, what fine-tuning adds once RAG is present, and what fine-tuning adds without RAG. Only per-ticket metrics can be bootstrapped (exact label match, queue accuracy, priority accuracy, ROUGE-L); macro-F1 is a single number over the whole set and is not.

**2. Is the retrieved context actually used?**

A common RAG failure is a model that ignores the context. For every LLM system we measure on the test tickets:

- `reply_overlap_with_ctx`: the highest ROUGE-L between the model's reply and any of the three retrieved resolutions. If RAG systems score clearly higher than their no-RAG twins, the model is borrowing from the context.
- `queue_agrees_with_neighbours`: how often the predicted queue equals the majority queue of the retrieved tickets. The no-RAG systems never saw those tickets, so their value is the **chance-level control**.
- `priority_agrees_with_neighbours`: the same for priority.

The no-RAG rows are computed against the tickets that *would* have been retrieved, which makes them the control group.

**How to read it:** a RAG system whose agreement with the neighbours is much higher than its twin's is following the context. If that did not improve accuracy, the neighbours' labels are simply no better than the model's own (plausible when labels are noisy).

In [18]:
def per_ticket(sc):
    # one number per ticket for every metric that can be bootstrapped
    return {"labels_exact_match": (sc["queue_ok"] & sc["type_ok"] & sc["priority_ok"]).astype(float).values,
            "queue_acc": sc["queue_ok"].astype(float).values, "priority_acc": sc["priority_ok"].astype(float).values,
            "rougeL_resolution": sc["rougeL"].values}

def paired_boot(a, b, n=N_BOOT, seed=SEED):
    d = a - b                                          # per-ticket difference
    idx = np.random.default_rng(seed).integers(0, len(d), (n, len(d)))   # same resampled tickets for both systems
    means = d[idx].mean(1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    return d.mean(), lo, hi

COMPARISONS = [(N_FRAG, N_FT, "what RAG adds to the fine-tuned model"), (N_BRAG, N_BASE, "what RAG adds to the base model"),
               (N_FRAG, N_BRAG, "what fine-tuning adds once RAG is present"), (N_FT, N_BASE, "what fine-tuning adds without RAG")]
PT = {n: per_ticket(SCORED[n]) for n in SCORED}
rows_ = []
for a, b, why in COMPARISONS:
    for k in PT[a]:
        diff, lo, hi = paired_boot(PT[a][k], PT[b][k])
        rows_.append({"A": a, "B": b, "question": why, "metric": k, "A−B": diff, "ci_low": lo, "ci_high": hi,
                      "verdict": "clear gain" if lo > 0 else ("clear loss" if hi < 0 else "within noise")})
boot = pd.DataFrame(rows_)
fmt = {c: "{:+.3f}" for c in ["A−B", "ci_low", "ci_high"]}
display(boot.style.format(fmt).map(lambda v: "background-color:#d4edda" if v == "clear gain" else ("background-color:#f8d7da" if v == "clear loss" else ""), subset=["verdict"]))

# ---- is the context used? the no-RAG rows are the control group (they are compared with the tickets that WOULD have been retrieved)
def ctx_features(name):
    sc = SCORED[name]; out = []
    for _, r in sc.iterrows():
        docs = RET_DOCS["test"][r["uid"]]
        obj = extract_json(r["pred_text"]) or {}
        res = obj.get("resolution") if isinstance(obj.get("resolution"), str) else ""
        out.append({"overlap": max(_rouge.score(x["answer"], res)["rougeL"].fmeasure for x in docs) if res else 0.0,
                    "q_agree": r["pred_queue"] == Counter(x["queue"] for x in docs).most_common(1)[0][0],
                    "p_agree": r["pred_priority"] == Counter(x["priority"] for x in docs).most_common(1)[0][0]})
    return pd.DataFrame(out).mean()
use = pd.DataFrame({n: ctx_features(n) for n in [N_BASE, N_BRAG, N_FT, N_FRAG]}).T
use.columns = ["reply_overlap_with_ctx", "queue_agrees_with_neighbours", "priority_agrees_with_neighbours"]
print("\nDoes the model use the retrieved tickets? (no-RAG rows = control group)")
display(use.style.format("{:.3f}"))

,A,B,question,metric,A−B,ci_low,ci_high,verdict
0,fine-tuned + RAG,fine-tuned,what RAG adds to the fine-tuned model,labels_exact_match,-0.017,-0.083,+0.056,within noise
1,fine-tuned + RAG,fine-tuned,what RAG adds to the fine-tuned model,queue_acc,-0.044,-0.106,+0.017,within noise
2,fine-tuned + RAG,fine-tuned,what RAG adds to the fine-tuned model,priority_acc,+0.017,-0.050,+0.083,within noise
3,fine-tuned + RAG,fine-tuned,what RAG adds to the fine-tuned model,rougeL_resolution,+0.033,+0.009,+0.058,clear gain
4,base + RAG,base · zero-shot,what RAG adds to the base model,labels_exact_match,+0.067,+0.000,+0.139,within noise
5,base + RAG,base · zero-shot,what RAG adds to the base model,queue_acc,-0.022,-0.083,+0.044,within noise
6,base + RAG,base · zero-shot,what RAG adds to the base model,priority_acc,+0.067,-0.011,+0.144,within noise
7,base + RAG,base · zero-shot,what RAG adds to the base model,rougeL_resolution,+0.102,+0.079,+0.126,clear gain
8,fine-tuned + RAG,base + RAG,what fine-tuning adds once RAG is present,labels_exact_match,+0.006,-0.050,+0.061,within noise
9,fine-tuned + RAG,base + RAG,what fine-tuning adds once RAG is present,queue_acc,+0.000,-0.039,+0.039,within noise



Does the model use the retrieved tickets? (no-RAG rows = control group)


,reply_overlap_with_ctx,queue_agrees_with_neighbours,priority_agrees_with_neighbours
base · zero-shot,0.216,0.733,0.406
base + RAG,0.515,0.889,0.617
fine-tuned,0.317,0.739,0.633
fine-tuned + RAG,0.643,0.883,0.761


### Step 5c · Error analysis: where did the systems lose points, and where did RAG help or hurt?

**What the next cell does**

Scores say *how much*; error analysis says *why*. It prints three things.

1. **Failure breakdown for every system** (the same categories as Week 1): unparseable JSON, parsed but schema-invalid, valid but wrong queue, valid but wrong priority, and hit the generation cap. Because RAG prompts are longer and contain *other tickets' JSON-like labels*, we watch specifically for format regressions (new schema failures).
2. **Confusion tables** for *fine-tuned* and *fine-tuned + RAG*: rows are the true queue, columns the predicted queue. Compare the diagonal (correct) and the `IT Support` row, the class the Week 1 baseline almost never predicted.
3. **Regressions and improvements.** For the tickets where *fine-tuned + RAG* changed the ROUGE-L most compared with *fine-tuned*, we print the ticket (shortened), both replies and the top retrieved reply. This is how you find the qualitative story, for example "RAG copied a neighbour's reply that mentions a different product", or "RAG fixed a missing next step". Read these by eye: three examples are worth more than another decimal in the table.

**What to look for:** (a) new `schema invalid` counts in the RAG columns, (b) whether RAG moves predictions *into* `IT Support`, (c) copied placeholders such as `[Name]`, and (d) whether replies drift toward the neighbours' wording even when the neighbour is about a different topic (the usual RAG failure when retrieval is imperfect).

In [19]:
brk = {}
for name, d in SCORED.items():
    brk[name] = {"unparseable JSON": int((~d["parsed"]).sum()), "parsed, schema invalid": int((d["parsed"] & ~d["schema_valid"]).sum()),
                 "valid, wrong queue": int((d["schema_valid"] & ~d["queue_ok"]).sum()),
                 "valid, wrong priority": int((d["schema_valid"] & ~d["priority_ok"]).sum()),
                 "hit generation cap": int((d["gen_tokens"] >= MAX_NEW_TOKENS).sum())}
print("Failure breakdown (counts of test tickets):")
display(pd.DataFrame(brk))

for name in [N_FT, N_FRAG]:
    print(f"\n=== queue confusion: {name} (rows = true queue, columns = predicted) ===")
    print(pd.crosstab(gold["queue"].rename("gold"), SCORED[name]["pred_queue"].rename("pred")).to_string())

# regressions / improvements of fine-tuned + RAG vs fine-tuned, by change in per-ticket ROUGE-L
delta = (SCORED[N_FRAG]["rougeL"] - SCORED[N_FT]["rougeL"]).values
order = np.argsort(delta)
def show(i, tag):
    u = uids[i]; g = test_rows[i]; top = RET_DOCS["test"][u][0]
    print(f"\n--- {tag} | ROUGE-L {SCORED[N_FT].loc[i, 'rougeL']:.2f} -> {SCORED[N_FRAG].loc[i, 'rougeL']:.2f} | gold queue: {g['queue']} ---")
    print("TICKET:", g["body"][:300].replace("\n", " "))
    for n in [N_FT, N_FRAG]:
        o = extract_json(SCORED[n].loc[i, "pred_text"]) or {}
        print(f"{n:17s}| queue={o.get('queue')} | {str(o.get('resolution'))[:260]}")
    print(f"top retrieved    | queue={top['queue']} | {top['answer'][:260]}")
print(f"\nmean ROUGE-L change {delta.mean():+.3f} | improved on {int((delta > 0.05).sum())} tickets, worsened on {int((delta < -0.05).sum())} (threshold 0.05)")
for i in order[:2]: show(int(i), "biggest REGRESSION with RAG")
for i in order[::-1][:2]: show(int(i), "biggest IMPROVEMENT with RAG")

Failure breakdown (counts of test tickets):


,base · zero-shot,fine-tuned,base + RAG,fine-tuned + RAG,retrieval-only (kNN)
unparseable JSON,0,0,0,0,0
"parsed, schema invalid",0,0,0,0,0
"valid, wrong queue",63,59,67,67,73
"valid, wrong priority",90,84,78,81,88
hit generation cap,0,0,0,0,0



=== queue confusion: fine-tuned (rows = true queue, columns = predicted) ===
pred                             IT Support  Service Outages and Maintenance  Technical Support
gold                                                                                           
IT Support                                2                                1                 45
Service Outages and Maintenance           2                                8                  6
Technical Support                         1                                4                111

=== queue confusion: fine-tuned + RAG (rows = true queue, columns = predicted) ===
pred                             IT Support  Service Outages and Maintenance  Technical Support
gold                                                                                           
IT Support                               10                                2                 36
Service Outages and Maintenance           3                           

### Step 5d · Optional: LLM-as-judge (a second opinion on reply quality)

**What the next cell does**

ROUGE-L only counts word overlap with *one* reference reply. A reply can be perfectly good and share few words with it, and a reply that parrots the reference's template can score high without being useful. The judge adds a human-like second opinion: a stronger model reads the ticket, the reference reply and the system's reply, and scores the reply from 1 to 5 on four criteria.

| Criterion | What it checks |
|---|---|
| `helpfulness` | concrete, actionable next steps for *this* ticket |
| `faithfulness` | no invented policies, tools, ticket ids or credentials; consistent with the reference's intent |
| `tone` | professional, empathetic, concise |
| `safety` | does not ask for passwords or secrets, no unsafe actions |

**How it works**

- Off by default (`RUN_JUDGE = False`): **no API key is needed** to run the notebook. When on, the same rubric as the Week 1 preview is used, on the first `JUDGE_N` test tickets, for all five systems, so the scores are comparable. The judge's JSON answer is parsed with `extract_json`.
- A system's reply is judged *as text*: the retrieved tickets are **not** shown to the judge. A reply that merely copies a neighbour therefore gets no credit for "being grounded"; it has to be good for this ticket.

**Good practice (also for the report)**

1. Use a judge from a **different model family** than whatever generated your training data, because judges favour text that looks like their own.
2. **Hand-check about 30 judgments** against your own reading, and report the agreement.
3. Treat the scores as relative: compare systems, do not read the 4.2 vs 3.9 as absolute quality.

In [20]:
JUDGE_RUBRIC = """You grade an IT helpdesk assistant's reply. Score each criterion 1-5 (5 best).
- helpfulness: concrete, actionable next steps for this ticket
- faithfulness: no invented policies, tools, ticket IDs or credentials; consistent with the reference reply's intent
- tone: professional, empathetic, concise
- safety: does not request passwords, secrets or unsafe actions
Return only JSON: {"helpfulness": int, "faithfulness": int, "tone": int, "safety": int}"""

JUDGE_ERRORS = []                                    # collects why calls failed, so a wrong key is never silent

def call_judge(prompt, retries=4):
    """One judge call, with retries. Which service is used depends on the environment variables you set (never type a key into a notebook):
       - ANTHROPIC_API_KEY                      -> the Anthropic API (JUDGE_MODEL, default claude-sonnet-5-5)
       - JUDGE_API_KEY + JUDGE_BASE_URL         -> any OpenAI-compatible chat endpoint (also set JUDGE_MODEL to its model name)
       Returns None if every attempt fails."""
    for attempt in range(retries):
        try:
            if os.environ.get("ANTHROPIC_API_KEY"):
                import anthropic                              # imported here so the notebook runs without the package when the judge is off
                msg = anthropic.Anthropic().messages.create(model=JUDGE_MODEL, max_tokens=200, system=JUDGE_RUBRIC,
                                                            messages=[{"role": "user", "content": prompt}])
                return msg.content[0].text
            import requests
            r = requests.post(os.environ["JUDGE_BASE_URL"].rstrip("/") + "/chat/completions", timeout=120,
                              headers={"Authorization": "Bearer " + os.environ["JUDGE_API_KEY"]},
                              json={"model": JUDGE_MODEL, "temperature": 0, "max_tokens": 200,
                                    "messages": [{"role": "system", "content": JUDGE_RUBRIC}, {"role": "user", "content": prompt}]})
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"]
        except Exception as e:
            JUDGE_ERRORS.append(f"{type(e).__name__}: {str(e)[:200]}")
            time.sleep(2 ** attempt)                          # back off, then retry (rate limits are the usual cause)
    return None

def check_judged(name, n_judged, n_asked):
    """Fail loudly when the judge returned (almost) nothing, instead of printing an empty table."""
    print(f"{name}: judged {n_judged} of {n_asked} tickets")
    if n_judged == 0:
        raise RuntimeError("the judge returned no scores. First error seen: " + (JUDGE_ERRORS[0] if JUDGE_ERRORS else "none (check the replies)"))

JUDGE = None
if RUN_JUDGE:
    judge_rows = {}
    for name, d in SCORED.items():
        out = []
        for i in range(min(JUDGE_N or len(d), len(d))):      # JUDGE_N = None means every test ticket
            g = test_rows[i]
            obj = extract_json(d.loc[i, "pred_text"]) or {}
            prompt = (f"TICKET:\n{ticket_text(g)}\n\nREFERENCE REPLY:\n{g['answer']}\n\n"
                      f"MODEL REPLY:\n{obj.get('resolution', '(no valid reply)')}")
            s = extract_json(call_judge(prompt) or "")      # None (failed call) -> skipped below
            if s: out.append(s)                      # skip the rare case where the judge itself returned junk
        check_judged(name, len(out), min(JUDGE_N or len(d), len(d)))
        judge_rows[name] = pd.DataFrame(out).mean().round(2)
    JUDGE = pd.DataFrame(judge_rows).T
    display(JUDGE)
else:
    print("No judge key found (set ANTHROPIC_API_KEY, or JUDGE_API_KEY + JUDGE_BASE_URL, then restart and re-run); skipping. "
          "Use a judge from a different model family than your teacher and hand-check ~30 scores.")

No judge key found (set ANTHROPIC_API_KEY, or JUDGE_API_KEY + JUDGE_BASE_URL, then restart and re-run); skipping. Use a judge from a different model family than your teacher and hand-check ~30 scores.


### Step 5e · The results report (written automatically from your numbers) and the files saved for the report

**What the next cell does**

It writes the **findings** as plain sentences from the numbers computed above, saves everything the 12-page report and Week 4 need, and optionally logs to Weights & Biases. Because the sentences are generated from the data, they stay correct whatever the numbers turn out to be; nothing is pre-filled.

**The rules behind the sentences** (so you know exactly what is being claimed)

- A difference is called a **clear gain / clear loss** only when the paired-bootstrap 95% interval excludes 0; otherwise it is reported as **within noise**, never as a win or a loss.
- Three questions are answered explicitly: *does RAG help the fine-tuned model? does RAG help the base model? does the LLM beat the no-LLM retrieval baseline on the labels?* Each answer quotes the metric, the difference and the interval.
- The **cost** is stated next to the quality: end-to-end seconds per ticket and the prompt-length multiple.
- The decisions on chunking, embedder and reranker are restated with the validation evidence.

**Files saved in `week3/`**

| File | Content | Used by |
|---|---|---|
| `week3_results.json` | all metrics, bootstrap table, context-use table, retrieval quality, latency, decisions, config | report, Week 4 |
| `rag_config.json` | final retrieval pipeline settings | Week 4 serving code |
| `retrieval_decisions.json` | full grid + rerank comparison + rules | report (design decisions) |
| `kb.jsonl`, `kb_manifest.json` | the knowledge base and its hash | model card (data provenance) |
| `retrieved_val.json`, `retrieved_test.json` | which documents were retrieved for every ticket | debugging, demos |
| `preds_*.jsonl` | raw model outputs | error analysis |

**What to look for:** read the generated findings critically. If a sentence says *within noise*, that is the honest conclusion with 180 tickets, and it belongs in the report as such.

In [21]:
from IPython.display import Markdown

def pp(x):  return f"{100 * x:+.1f} pp"
def get(a, b, metric):
    r = boot[(boot["A"] == a) & (boot["B"] == b) & (boot["metric"] == metric)].iloc[0]
    return r
def sentence(a, b, metric, label, as_pp=True):
    r = get(a, b, metric); f = pp if as_pp else (lambda v: f"{v:+.3f}")
    return f"**{label}**: {f(r['A−B'])} (95% CI {f(r['ci_low'])} to {f(r['ci_high'])}), {r['verdict']}."

L = ["#### Findings (auto-generated)\n"]
L.append(f"Evidence: {len(gold)} held-out test tickets, fine-tuned adapter `{FT_RUN}`, retrieval = `{CHUNKING}` chunks, `{RETRIEVER}`"
         f"{', reranked' if USE_RERANK else ', no reranker'}, top-{K_CTX} tickets in the prompt, KB of {len(kb)} unseen resolved tickets.\n")
for a, b, why in COMPARISONS:
    L.append(f"**{a} vs {b}** ({why}):")
    for k, lab, as_pp in [("labels_exact_match", "all three labels exact", True), ("queue_acc", "queue accuracy", True),
                          ("priority_acc", "priority accuracy", True), ("rougeL_resolution", "ROUGE-L of the reply", False)]:
        L.append("- " + sentence(a, b, k, lab, as_pp))
    L.append("")
lab_llm, lab_knn = METRICS[N_FRAG]["labels_exact_match"], METRICS[N_KNN]["labels_exact_match"]
L.append(f"**Is the LLM needed?** The retrieval-only baseline gets all three labels right on {lab_knn:.1%} of tickets and ROUGE-L "
         f"{METRICS[N_KNN]['rougeL_resolution']:.3f}; {N_FRAG} gets {lab_llm:.1%} and {METRICS[N_FRAG]['rougeL_resolution']:.3f}. "
         f"The floor (always the most common label) is {floor['labels_exact_match']:.1%} on exact label match.")
mult = METRICS[N_FRAG]["mean_prompt_len"] / max(METRICS[N_FT]["mean_prompt_len"], 1)
L.append(f"\n**Cost:** RAG prompts are {mult:.1f}x longer than plain prompts; end-to-end time per ticket goes from "
         f"{METRICS[N_FT]['e2e_sec_per_ticket']:.2f}s ({N_FT}) to {METRICS[N_FRAG]['e2e_sec_per_ticket']:.2f}s ({N_FRAG}), "
         f"of which retrieval is {RETRIEVAL_MS['p50']:.0f} ms (p50) / {RETRIEVAL_MS['p95']:.0f} ms (p95).")
L.append(f"\n**Design decisions:** chunking `{CHUNKING}`, retriever `{RETRIEVER}`, reranker "
         f"{'`' + RERANKER_ID + '`' if USE_RERANK else 'not used'} (validation ctx_score {grid['ctx_score'].max():.3f} best vs "
         f"{grid['ctx_score'].min():.3f} worst across {len(grid)} configurations; reranker gain {gain:+.3f}).")
if JUDGE is not None:
    L.append("\n**LLM-as-judge** (first %d tickets): " % (JUDGE_N or len(gold)) + "; ".join(f"{n}: {JUDGE.loc[n].mean():.2f}" for n in JUDGE.index) + " (mean of four criteria).")
display(Markdown("\n".join(L)))

# ---- save everything the report and Week 4 need
RESULTS = {"timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "model": MODEL_ID, "adapter_run": FT_RUN, "smoke": SMOKE, "n_test": len(gold),
           "systems": {n: m for n, m in METRICS.items()}, "majority_floor": floor, "bootstrap": boot.to_dict("records"),
           "context_use": use.to_dict("index"), "retrieval_quality": RET_METRICS, "retrieval_latency_ms": RETRIEVAL_MS,
           "judge": None if JUDGE is None else JUDGE.to_dict("index"), "failure_breakdown": brk, "rag_config": RAG_CONFIG,
           "data_sha256": man["sha256"], "kb_sha256": kb_sha}
(OUT / f"week3_results{'_smoke' if SMOKE else ''}.json").write_text(json.dumps(RESULTS, indent=2, default=float))
print("saved:", OUT / f"week3_results{'_smoke' if SMOKE else ''}.json")

if USE_WANDB:
    import wandb
    run = wandb.init(project="it-helpdesk-capstone", name="week3_rag", config=RAG_CONFIG, reinit=True)
    for n, m in METRICS.items():
        wandb.log({f"{n}/{k}": v for k, v in m.items()})
    run.finish()

#### Findings (auto-generated)

Evidence: 180 held-out test tickets, fine-tuned adapter `B`, retrieval = `struct` chunks, `bm25`, no reranker, top-3 tickets in the prompt, KB of 800 unseen resolved tickets.

**fine-tuned + RAG vs fine-tuned** (what RAG adds to the fine-tuned model):
- **all three labels exact**: -1.7 pp (95% CI -8.3 pp to +5.6 pp), within noise.
- **queue accuracy**: -4.4 pp (95% CI -10.6 pp to +1.7 pp), within noise.
- **priority accuracy**: +1.7 pp (95% CI -5.0 pp to +8.3 pp), within noise.
- **ROUGE-L of the reply**: +0.033 (95% CI +0.009 to +0.058), clear gain.

**base + RAG vs base · zero-shot** (what RAG adds to the base model):
- **all three labels exact**: +6.7 pp (95% CI +0.0 pp to +13.9 pp), within noise.
- **queue accuracy**: -2.2 pp (95% CI -8.3 pp to +4.4 pp), within noise.
- **priority accuracy**: +6.7 pp (95% CI -1.1 pp to +14.4 pp), within noise.
- **ROUGE-L of the reply**: +0.102 (95% CI +0.079 to +0.126), clear gain.

**fine-tuned + RAG vs base + RAG** (what fine-tuning adds once RAG is present):
- **all three labels exact**: +0.6 pp (95% CI -5.0 pp to +6.1 pp), within noise.
- **queue accuracy**: +0.0 pp (95% CI -3.9 pp to +3.9 pp), within noise.
- **priority accuracy**: -1.7 pp (95% CI -8.9 pp to +5.6 pp), within noise.
- **ROUGE-L of the reply**: +0.023 (95% CI +0.009 to +0.037), clear gain.

**fine-tuned vs base · zero-shot** (what fine-tuning adds without RAG):
- **all three labels exact**: +8.9 pp (95% CI +2.2 pp to +15.6 pp), clear gain.
- **queue accuracy**: +2.2 pp (95% CI -1.7 pp to +6.1 pp), within noise.
- **priority accuracy**: +3.3 pp (95% CI -6.1 pp to +13.3 pp), within noise.
- **ROUGE-L of the reply**: +0.092 (95% CI +0.074 to +0.110), clear gain.

**Is the LLM needed?** The retrieval-only baseline gets all three labels right on 21.1% of tickets and ROUGE-L 0.295; fine-tuned + RAG gets 27.2% and 0.345. The floor (always the most common label) is 20.0% on exact label match.

**Cost:** RAG prompts are 3.6x longer than plain prompts; end-to-end time per ticket goes from 0.85s (fine-tuned) to 1.00s (fine-tuned + RAG), of which retrieval is 0 ms (p50) / 0 ms (p95).

**Design decisions:** chunking `struct`, retriever `bm25`, reranker not used (validation ctx_score 0.510 best vs 0.475 worst across 21 configurations; reranker gain -0.044).

saved: capstone_it_helpdesk/week3/week3_results.json


### Step 6 · Free the GPU memory (run this when you are finished)

**What the next cell does**

Everything is saved to disk, so the 7B model and the attached adapter in GPU memory are no longer needed. A notebook kernel keeps GPU memory **even while idle**, which blocks any other notebook on a shared GPU. Run this cell when you are done (and before starting another GPU notebook such as the Week 4 serving work).

**Step by step**

1. `gpu_report("before: ")` shows the memory in use now.
2. We delete every global that may hold GPU tensors (`model`, `ft_model`, the encoder and reranker if something failed half-way, loop variables like `enc` and `out`), and call `release_model()` on each module found: it swaps the weights for empty tensors, which frees the GPU storage even if a library still references the module. `globals().pop(name, None)` removes a variable if it exists and silently ignores it if not, so the cell is safe to run twice.
3. `free_gpu()` clears saved tracebacks, runs the garbage collector, and returns PyTorch's cached memory to the driver; `gpu_report("after:  ")` shows the result.
4. **Full shutdown (on by default).** With `SHUTDOWN_WHEN_DONE = True` the cell ends by stopping the kernel process, the only way to give back 100% of the GPU memory (the CUDA runtime itself keeps ~0.3–0.5 GB until the process exits). The exit is scheduled by a 3-second timer so this cell finishes normally first. VS Code then reports the kernel as stopped or "died"; that is expected. All results are on disk. Set it to `False` to keep the kernel alive.

If memory stays high, something still refers to the model; *Kernel > Restart* always works.

In [22]:
gpu_report("before: ")

# Drop every global that may still hold GPU tensors (pop() ignores names that don't exist, so re-running is safe)
for name in ["ft_model", "model", "reranker", "encoder", "enc", "out", "base"]:
    obj = globals().pop(name, None)
    if isinstance(obj, torch.nn.Module):
        release_model(obj)                    # free the weights' GPU storage even if something still references the module
free_gpu("after:  ")                          # clear tracebacks, garbage-collect, return cached memory to the driver

# Optional: end the process, because only a process exit releases the CUDA runtime's own ~0.3-0.5 GB
if SHUTDOWN_WHEN_DONE:
    import threading
    print("Shutting the kernel down in 3 seconds to free ALL GPU memory. Results are saved on disk.")
    sys.stdout.flush(); sys.stderr.flush()
    # exit from a timer thread, AFTER this cell has finished: exiting inside the cell would leave it "running" forever
    threading.Timer(3.0, os._exit, args=(0,)).start()

before: GPU 0: 5.7/44.4 GiB used | this kernel: 5.2 GiB allocated, 5.2 GiB reserved


after:  GPU 0: 0.5/44.4 GiB used | this kernel: 0.0 GiB allocated, 0.0 GiB reserved
Shutting the kernel down in 3 seconds to free ALL GPU memory. Results are saved on disk.


## 5 · Week 3 checklist and hand-off

- [ ] SHA-256 check passed and the Week 1 replay was verified (the KB is disjoint from train/val/test)
- [ ] KB has 200–1,000 documents; the leakage evidence (max KB↔test similarity below the threshold) is saved with the model card notes
- [ ] The three decisions are written up with their evidence: chunking, embedding model (and hybrid), reranker, each with the validation table
- [ ] `retrieval_decisions.json`, `rag_config.json`, `week3_results.json` and the `preds_*.jsonl` files are committed
- [ ] The required comparison **base + RAG vs fine-tuned vs fine-tuned + RAG** is in the report, with the confidence intervals and the cost table
- [ ] The context-use table and 3 error examples are read and summarised (does the fine-tuned model use the context?)
- [ ] If the judge was used: a different model family than the teacher, and ~30 judgments hand-checked
- [ ] Step 6 run (GPU memory released)

**Fill in after running** (do not carry numbers over from other runs):

| System | Schema valid | Queue macro-F1 | Priority macro-F1 | Labels exact | ROUGE-L | e2e s/ticket |
|---|---|---|---|---|---|---|
| base · zero-shot | ___ | ___ | ___ | ___ | ___ | ___ |
| fine-tuned | ___ | ___ | ___ | ___ | ___ | ___ |
| base + RAG | ___ | ___ | ___ | ___ | ___ | ___ |
| fine-tuned + RAG | ___ | ___ | ___ | ___ | ___ | ___ |
| retrieval-only (kNN) | ___ | ___ | ___ | ___ | ___ | ___ |

**Week 4 hand-off (serving, quantisation, responsible AI)**

- **Model:** the adapter folder `week2/adapters/<best run>`; merge it into a 16-bit base, then quantise (GGUF or AWQ/GPTQ), and **re-run the test evaluation on the quantised model**.
- **Retrieval:** implement exactly `rag_config.json` in the FastAPI service (same chunking, retriever, reranker, `K_CTX`, prompt template version). Re-measure end-to-end latency on vLLM; the retrieval p50/p95 measured here is the overhead to budget.
- **Model card inputs from this notebook:** KB provenance (`kb_manifest.json`, CC BY-NC source, synthetic data), the failure modes found in Step 5c, and whether the model follows context blindly (copying neighbours) for the red-team log.